
# K-Means Clustering Baselines

## Objective

Establish baseline clustering results for the nine validated DT-CARGO feature representations.

## Research questions

1. How does the choice of feature representation affect K-Means clustering?
2. Does dimensionality reduction improve or degrade cluster separation?
3. Do the original-feature and full-PCA representations produce equivalent clustering geometry?
4. Does adding telemetry-quality information produce useful operational segments or primarily recording-quality segments?

## Experimental constraints

- Use the same 42,867 tracks for all representations.
- Use K-Means with consistent initialization settings.
- Evaluate identical candidate values of k.
- Record runtime, silhouette score, cluster sizes, and random seeds.
- Do not choose a winner using silhouette score alone.


In [1]:

from pathlib import Path
import time
import numpy as np
import pandas as pd
import sklearn

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PROJECT_ROOT = Path.cwd().parent
INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "pca_clustering_inputs"

# Load the original standardized feature representations
original_candidates = {
    "Operational_Original": pd.read_parquet(
        INPUT_DIR / "Operational_Original.parquet"
    ),
    "Telemetry_Original": pd.read_parquet(
        INPUT_DIR / "Telemetry_Original.parquet"
    ),
    "Broad_Original": pd.read_parquet(
        INPUT_DIR / "Broad_Original.parquet"
    ),
}

# Load the PCA representations
pca_candidates = {
    "Operational_2PC": pd.read_parquet(
        INPUT_DIR / "Operational_2PC.parquet"
    ),
    "Operational_5PC": pd.read_parquet(
        INPUT_DIR / "Operational_5PC.parquet"
    ),
    "Telemetry_4PC": pd.read_parquet(
        INPUT_DIR / "Telemetry_4PC.parquet"
    ),
    "Telemetry_7PC": pd.read_parquet(
        INPUT_DIR / "Telemetry_7PC.parquet"
    ),
    "Broad_6PC": pd.read_parquet(
        INPUT_DIR / "Broad_6PC.parquet"
    ),
    "Broad_10PC": pd.read_parquet(
        INPUT_DIR / "Broad_10PC.parquet"
    ),
}

candidates = {**original_candidates, **pca_candidates}

print("scikit-learn version:", sklearn.__version__)
print("Number of representations:", len(candidates))

for name, X in candidates.items():
    print(f"{name:25} | shape={X.shape}")


scikit-learn version: 1.9.1
Number of representations: 9
Operational_Original      | shape=(42867, 5)
Telemetry_Original        | shape=(42867, 7)
Broad_Original            | shape=(42867, 10)
Operational_2PC           | shape=(42867, 2)
Operational_5PC           | shape=(42867, 5)
Telemetry_4PC             | shape=(42867, 4)
Telemetry_7PC             | shape=(42867, 7)
Broad_6PC                 | shape=(42867, 6)
Broad_10PC                | shape=(42867, 10)


In [2]:

reference_index = next(iter(candidates.values())).index

assert len(candidates) == 9, "Expected nine representations."

for name, X in candidates.items():
    assert X.shape[0] == 42867, f"Unexpected row count: {name}"
    assert X.index.is_unique, f"Duplicate index: {name}"
    assert X.index.equals(reference_index), f"Index mismatch: {name}"
    assert np.isfinite(X.to_numpy()).all(), f"Non-finite values: {name}"

print("All nine representations passed validation.")
print("Common tracks:", len(reference_index))


All nine representations passed validation.
Common tracks: 42867



## K-Means Baseline Experiment

### Candidate values of k

We will evaluate k = 2, 3, 4, 5, 6, 7, 8, and 10.

### Initialization controls

- Algorithm: K-Means
- Initialization: k-means++
- Number of initializations: 10
- Random seed: 42
- Maximum iterations: 300
- Evaluation metric: silhouette score using Euclidean distance

### Comparison controls

All nine representations use the same 42,867 tracks and the same candidate values of k. We will record runtime, silhouette score, cluster sizes, and convergence information.

The best candidate will not be selected by silhouette score alone.


In [3]:

# First baseline: operational features, original standardized space

representation_name = "Operational_Original"
X = candidates[representation_name].to_numpy()

k_values = [2, 3, 4, 5, 6, 7, 8, 10]
baseline_results = []
baseline_labels = {}

for k in k_values:
    model = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=42,
        algorithm="lloyd"
    )

    start_time = time.perf_counter()
    labels = model.fit_predict(X)
    elapsed = time.perf_counter() - start_time

    score = silhouette_score(
        X,
        labels,
        metric="euclidean"
    )

    cluster_sizes = pd.Series(labels).value_counts().sort_index()

    baseline_results.append({
        "representation": representation_name,
        "k": k,
        "inertia": model.inertia_,
        "silhouette": score,
        "runtime_seconds": elapsed,
        "iterations": model.n_iter_,
        "smallest_cluster": int(cluster_sizes.min()),
        "largest_cluster": int(cluster_sizes.max()),
        "cluster_sizes": cluster_sizes.to_dict()
    })

    baseline_labels[k] = labels

    print(
        f"k={k:2} | silhouette={score:.4f} | "
        f"inertia={model.inertia_:.2f} | "
        f"runtime={elapsed:.2f}s | "
        f"iterations={model.n_iter_}"
    )

baseline_results_df = pd.DataFrame(baseline_results)


k= 2 | silhouette=0.4872 | inertia=88878.06 | runtime=0.10s | iterations=4
k= 3 | silhouette=0.4031 | inertia=60391.63 | runtime=0.06s | iterations=8
k= 4 | silhouette=0.3649 | inertia=48284.88 | runtime=0.05s | iterations=7
k= 5 | silhouette=0.3521 | inertia=42719.56 | runtime=0.12s | iterations=20
k= 6 | silhouette=0.3157 | inertia=38311.80 | runtime=0.14s | iterations=41
k= 7 | silhouette=0.3222 | inertia=35110.60 | runtime=0.14s | iterations=19
k= 8 | silhouette=0.3076 | inertia=32019.69 | runtime=0.12s | iterations=19
k=10 | silhouette=0.3158 | inertia=27143.30 | runtime=0.15s | iterations=17


In [4]:

# Cluster-size distributions for the first baseline

cluster_size_rows = []

for result in baseline_results:
    k = result["k"]
    sizes = result["cluster_sizes"]

    row = {
        "k": k,
        "silhouette": result["silhouette"],
        "smallest_cluster": result["smallest_cluster"],
        "largest_cluster": result["largest_cluster"],
    }

    for cluster_id, size in sizes.items():
        row[f"cluster_{cluster_id}_count"] = size
        row[f"cluster_{cluster_id}_pct"] = round(
            100 * size / len(reference_index), 2
        )

    cluster_size_rows.append(row)

cluster_sizes_df = pd.DataFrame(cluster_size_rows)

display(cluster_sizes_df)


,k,silhouette,smallest_cluster,largest_cluster,cluster_0_count,cluster_0_pct,cluster_1_count,cluster_1_pct,cluster_2_count,cluster_2_pct,...,cluster_5_count,cluster_5_pct,cluster_6_count,cluster_6_pct,cluster_7_count,cluster_7_pct,cluster_8_count,cluster_8_pct,cluster_9_count,cluster_9_pct
0,2,0.487209,19121,23746,23746,55.39,19121,44.61,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3,0.403069,11040,16398,16398,38.25,15429,35.99,11040.0,25.75,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,0.364908,7481,12844,12844,29.96,10477,24.44,12065.0,28.15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,0.352055,4821,11440,7361,17.17,10495,24.48,8750.0,20.41,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,0.315710,4041,9113,4041,9.43,7918,18.47,9113.0,21.26,...,6671.0,15.56,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,7,0.322215,3960,8175,5543,12.93,8175,19.07,7068.0,16.49,...,4555.0,10.63,6432.0,15.00,NaN,NaN,NaN,NaN,NaN,NaN
6,8,0.307588,3082,8073,6208,14.48,3710,8.65,3082.0,7.19,...,8073.0,18.83,5431.0,12.67,4419.0,10.31,NaN,NaN,NaN,NaN
7,10,0.315849,84,8069,6228,14.53,7066,16.48,4426.0,10.32,...,3040.0,7.09,8069.0,18.82,84.0,0.20,4495.0,10.49,595.0,1.39


In [5]:

# Refit the k=2 baseline and attach cluster labels to track indices

X_original = candidates["Operational_Original"]

kmeans_2 = KMeans(
    n_clusters=2,
    init="k-means++",
    n_init=10,
    max_iter=300,
    random_state=42,
    algorithm="lloyd"
)

labels_2 = kmeans_2.fit_predict(X_original.to_numpy())

cluster_labels_2 = pd.Series(
    labels_2,
    index=X_original.index,
    name="cluster"
)

# Summarize each cluster in the standardized feature space
cluster_profile = X_original.groupby(cluster_labels_2).mean().T

print("Mean standardized feature values by cluster:")
display(cluster_profile)

print("\nCluster sizes:")
display(cluster_labels_2.value_counts().sort_index().rename("track_count").to_frame())


Mean standardized feature values by cluster:


cluster,0,1
distance_log,0.752805,-0.934894
duration_log,0.618533,-0.768144
raw_speed_mean_kmh,0.732129,-0.909217
max_speed_kmh,0.601410,-0.746879
avg_speed_kmh_for_pca,0.713881,-0.886555



Cluster sizes:


,track_count
cluster,
0,23746
1,19121


In [6]:

# K-Means baseline sweep across all nine representations

k_values = [2, 3, 4, 5, 6, 7, 8, 10]

all_results = []
all_labels = {}

for representation_name, X_df in candidates.items():
    X = X_df.to_numpy()

    print(f"\nRunning: {representation_name}")

    for k in k_values:
        model = KMeans(
            n_clusters=k,
            init="k-means++",
            n_init=10,
            max_iter=300,
            random_state=42,
            algorithm="lloyd"
        )

        start_time = time.perf_counter()
        labels = model.fit_predict(X)
        fit_runtime = time.perf_counter() - start_time

        # Exact silhouette score on all 42,867 tracks
        start_time = time.perf_counter()
        silhouette = silhouette_score(
            X,
            labels,
            metric="euclidean"
        )
        silhouette_runtime = time.perf_counter() - start_time

        cluster_sizes = pd.Series(labels).value_counts()

        result = {
            "representation": representation_name,
            "k": k,
            "inertia": model.inertia_,
            "silhouette": silhouette,
            "fit_runtime_seconds": fit_runtime,
            "silhouette_runtime_seconds": silhouette_runtime,
            "iterations": model.n_iter_,
            "smallest_cluster": int(cluster_sizes.min()),
            "largest_cluster": int(cluster_sizes.max()),
            "smallest_cluster_pct": (
                100 * cluster_sizes.min() / len(X_df)
            ),
            "largest_cluster_pct": (
                100 * cluster_sizes.max() / len(X_df)
            ),
            "cluster_sizes": cluster_sizes.to_dict()
        }

        all_results.append(result)

        # Preserve labels indexed by original track index
        all_labels[(representation_name, k)] = pd.Series(
            labels,
            index=X_df.index,
            name="cluster"
        )

        print(
            f"  k={k:2} | silhouette={silhouette:.4f} | "
            f"inertia={model.inertia_:.2f} | "
            f"fit={fit_runtime:.2f}s | "
            f"silhouette_time={silhouette_runtime:.2f}s"
        )

all_results_df = pd.DataFrame(all_results)

print("\nCompleted baseline runs:", len(all_results_df))



Running: Operational_Original
  k= 2 | silhouette=0.4872 | inertia=88878.06 | fit=0.04s | silhouette_time=8.01s
  k= 3 | silhouette=0.4031 | inertia=60391.63 | fit=0.06s | silhouette_time=6.99s
  k= 4 | silhouette=0.3649 | inertia=48284.88 | fit=0.05s | silhouette_time=6.53s
  k= 5 | silhouette=0.3521 | inertia=42719.56 | fit=0.12s | silhouette_time=6.44s
  k= 6 | silhouette=0.3157 | inertia=38311.80 | fit=0.14s | silhouette_time=6.25s
  k= 7 | silhouette=0.3222 | inertia=35110.60 | fit=0.14s | silhouette_time=6.05s
  k= 8 | silhouette=0.3076 | inertia=32019.69 | fit=0.13s | silhouette_time=6.01s
  k=10 | silhouette=0.3158 | inertia=27143.30 | fit=0.15s | silhouette_time=6.02s

Running: Telemetry_Original
  k= 2 | silhouette=0.4146 | inertia=164533.71 | fit=0.04s | silhouette_time=8.27s
  k= 3 | silhouette=0.3054 | inertia=134820.54 | fit=0.06s | silhouette_time=7.34s
  k= 4 | silhouette=0.3141 | inertia=106130.45 | fit=0.10s | silhouette_time=7.35s
  k= 5 | silhouette=0.2750 | inerti

In [7]:

from pathlib import Path
from sklearn.metrics import adjusted_rand_score

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "clustering_baselines"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Save numeric metrics
metrics_to_save = all_results_df.drop(columns=["cluster_sizes"])
metrics_to_save.to_csv(
    OUTPUT_DIR / "kmeans_baseline_metrics.csv",
    index=False
)

# Save cluster-size dictionaries
all_results_df[
    ["representation", "k", "cluster_sizes"]
].to_json(
    OUTPUT_DIR / "kmeans_cluster_sizes.json",
    orient="records",
    indent=2
)

# Save all cluster assignments with the original track index
labels_df = pd.DataFrame(
    {
        f"{name}_k{k}": labels
        for (name, k), labels in all_labels.items()
    },
    index=reference_index
)

labels_df.to_parquet(
    OUTPUT_DIR / "kmeans_cluster_labels.parquet",
    index=True
)

print("Saved baseline results to:", OUTPUT_DIR.resolve())
print("Metrics:", metrics_to_save.shape)
print("Labels:", labels_df.shape)

# Verify full-PCA and original-feature clustering equivalence
equivalence_pairs = [
    ("Operational_Original", "Operational_5PC"),
    ("Telemetry_Original", "Telemetry_7PC"),
    ("Broad_Original", "Broad_10PC"),
]

print("\nAdjusted Rand Index for full-PCA comparisons (k=2):")

for original_name, pca_name in equivalence_pairs:
    ari = adjusted_rand_score(
        all_labels[(original_name, 2)],
        all_labels[(pca_name, 2)]
    )
    print(f"{original_name:25} vs {pca_name:20} | ARI={ari:.6f}")


Saved baseline results to: /Users/janibhargesh/Documents/ML-Project/data/processed/clustering_baselines
Metrics: (72, 11)
Labels: (42867, 72)

Adjusted Rand Index for full-PCA comparisons (k=2):
Operational_Original      vs Operational_5PC      | ARI=1.000000
Telemetry_Original        vs Telemetry_7PC        | ARI=1.000000
Broad_Original            vs Broad_10PC           | ARI=1.000000


In [8]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "clustering_baselines"

# Load the saved K-Means labels
labels_df = pd.read_parquet(
    OUTPUT_DIR / "kmeans_cluster_labels.parquet"
)

# Load the experiment metrics
metrics_df = pd.read_csv(
    OUTPUT_DIR / "kmeans_baseline_metrics.csv"
)

# Load the engineered track dataset
tracks_df = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "engineered_tracks.csv"
)

print("Labels shape:", labels_df.shape)
print("Metrics shape:", metrics_df.shape)
print("Engineered tracks shape:", tracks_df.shape)

print("\nEngineered dataset columns:")
print(tracks_df.columns.tolist())

print("\nLabel index sample:")
print(labels_df.index[:5].tolist())

print("\nEngineered dataset index sample:")
print(tracks_df.index[:5].tolist())

print("\nTrack index alignment:")
print(
    "Same index:",
    labels_df.index.equals(tracks_df.index)
)


Labels shape: (42867, 72)
Metrics shape: (72, 11)
Engineered tracks shape: (151598, 63)

Engineered dataset columns:
['track_id', 'vehicle_id', 'tour_id', 'start_time', 'stop_time', 'distance', 'track_gap', 'avg_speed', 'max_speed', 'n_signal_loss', 'd_signal_loss', 'r_signal_loss', 'avg_hdop', 'home_base', 'long_haul', 'rest_area', 'service_area_fuel', 'industrial_area', 'cid', 'fleet_test_id', 'gross_vehicle_weight', 'total_mass_with_trailer', 'axle_class', 'raw_observations', 'telemetry_span_seconds', 'raw_speed_mean', 'raw_speed_std', 'raw_speed_min', 'raw_speed_max', 'raw_hdop_mean', 'raw_hdop_median', 'raw_hdop_max', 'missing_speed_count', 'missing_hdop_count', 'insufficient_telemetry', 'start_time_parsed', 'stop_time_parsed', 'duration_seconds', 'duration_invalid', 'avg_speed_negative', 'avg_speed_gt_max_speed', 'raw_speed_missing', 'n_signal_loss_negative', 'r_signal_loss_missing', 'r_signal_loss_out_of_range', 'distance_missing', 'zero_distance', 'zero_distance_speed_issue', '

In [9]:

# Select the engineered rows corresponding to the clustering labels.
# .loc uses the saved DataFrame index, not row position.

aligned_tracks = tracks_df.loc[labels_df.index].copy()

print("Aligned tracks shape:", aligned_tracks.shape)
print("Labels shape:", labels_df.shape)
print(
    "Index alignment:",
    aligned_tracks.index.equals(labels_df.index)
)

# Check that every selected index exists in the engineered dataset.
missing_indices = labels_df.index.difference(tracks_df.index)

print("Missing indices:", len(missing_indices))

# Add cluster assignments for the leading candidate at k=2.
aligned_tracks["operational_2pc_k2"] = labels_df["Operational_2PC_k2"]

print("\nCluster counts:")
print(
    aligned_tracks["operational_2pc_k2"]
    .value_counts()
    .sort_index()
)

print("\nCluster proportions (%):")
print(
    (
        aligned_tracks["operational_2pc_k2"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
    ).round(2)
)


Aligned tracks shape: (42867, 63)
Labels shape: (42867, 72)
Index alignment: True
Missing indices: 0

Cluster counts:
operational_2pc_k2
0    19036
1    23831
Name: count, dtype: int64

Cluster proportions (%):
operational_2pc_k2
0    44.41
1    55.59
Name: proportion, dtype: float64


In [10]:

# Original-unit features for interpreting the two clusters
profile_features = [
    "distance_km",
    "duration_minutes",
    "avg_speed_kmh",
    "max_speed_kmh",
    "raw_speed_mean_kmh",
]

# Check feature availability
missing_features = [
    col for col in profile_features
    if col not in aligned_tracks.columns
]
print("Missing profile features:", missing_features)

# Calculate median, mean, and quartiles for each cluster
cluster_profile = (
    aligned_tracks
    .groupby("operational_2pc_k2")[profile_features]
    .agg(["median", "mean", "std", "min", "max"])
    .round(2)
)

display(cluster_profile)

# Also compare cluster medians with the overall median
overall_medians = aligned_tracks[profile_features].median().rename("Overall")
cluster_medians = (
    aligned_tracks
    .groupby("operational_2pc_k2")[profile_features]
    .median()
    .T
)

median_comparison = cluster_medians.join(overall_medians)
display(median_comparison.round(2))


Missing profile features: []


distance_km                             duration_minutes  \
                        median   mean    std   min     max           median   
operational_2pc_k2                                                            
0                         4.87   6.75   6.27  1.00  153.09            19.13   
1                        63.01  85.40  71.44  1.04  476.55            72.70   

                                                 ... max_speed_kmh         \
                     mean    std   min      max  ...        median   mean   
operational_2pc_k2                               ...                        
0                   26.50  26.61  1.00   500.46  ...         66.13  59.88   
1                   90.87  61.90  1.54  1029.73  ...         90.43  88.77   

                                         raw_speed_mean_kmh                \
                      std    min     max             median   mean    std   
operational_2pc_k2                                                          
0                   19.88   0.36  165.71              16.17  17.65  11.52   
1                   11.47  38.99  455.94              55.13  54.88  14.34   

                                  
                     min     max  
operational_2pc_k2                
0                   0.07   88.78  
1                   1.07  241.74  

[2 rows x 25 columns]

,0,1,Overall
distance_km,4.87,63.01,25.66
duration_minutes,19.13,72.70,42.55
avg_speed_kmh,16.31,54.43,37.98
max_speed_kmh,66.13,90.43,82.66
raw_speed_mean_kmh,16.17,55.13,38.87


In [11]:

# Examine the existing metadata against the two clusters.
metadata_features = [
    "vehicle_id",
    "fleet_test_id",
    "long_haul",
    "home_base",
    "rest_area",
    "service_area_fuel",
    "industrial_area",
]

available_metadata = [
    col for col in metadata_features
    if col in aligned_tracks.columns
]

print("Metadata available:", available_metadata)

for col in available_metadata:
    print(f"\n--- {col} ---")

    if aligned_tracks[col].nunique(dropna=False) <= 20:
        counts = pd.crosstab(
            aligned_tracks["operational_2pc_k2"],
            aligned_tracks[col],
            normalize="index",
            dropna=False,
        ).mul(100).round(2)

        print("Percentage within each cluster:")
        display(counts)
    else:
        summary = (
            aligned_tracks
            .groupby("operational_2pc_k2")[col]
            .nunique()
            .rename("unique_values")
        )
        print(summary)


Metadata available: ['vehicle_id', 'fleet_test_id', 'long_haul', 'home_base', 'rest_area', 'service_area_fuel', 'industrial_area']

--- vehicle_id ---
operational_2pc_k2
0    150
1    147
Name: unique_values, dtype: int64

--- fleet_test_id ---
Percentage within each cluster:


fleet_test_id,1,2,3,4,5,6
operational_2pc_k2,,,,,,
0,14.09,12.10,10.70,9.66,18.40,35.05
1,17.68,5.48,7.72,14.95,23.59,30.58



--- long_haul ---
Percentage within each cluster:


long_haul,False,True
operational_2pc_k2,,
0,93.29,6.71
1,83.53,16.47



--- home_base ---
Percentage within each cluster:


...

rest_area,False,True
operational_2pc_k2,,
0,99.74,0.26
1,95.03,4.97



--- service_area_fuel ---
Percentage within each cluster:


service_area_fuel,False,True
operational_2pc_k2,,
0,99.70,0.30
1,95.04,4.96



--- industrial_area ---
Percentage within each cluster:


industrial_area,False,True
operational_2pc_k2,,
0,48.21,51.79
1,47.10,52.90


In [12]:

# Count tracks from each vehicle in each cluster
vehicle_cluster_counts = pd.crosstab(
    aligned_tracks["vehicle_id"],
    aligned_tracks["operational_2pc_k2"]
)

# Include zero-count columns if necessary
vehicle_cluster_counts = vehicle_cluster_counts.reindex(
    columns=[0, 1],
    fill_value=0
)

# Determine whether each vehicle contributes to one or both clusters
vehicle_cluster_counts["clusters_present"] = (
    (vehicle_cluster_counts[[0, 1]] > 0).sum(axis=1)
)

vehicles_both = (
    vehicle_cluster_counts["clusters_present"] == 2
).sum()

vehicles_one = (
    vehicle_cluster_counts["clusters_present"] == 1
).sum()

print("Vehicles represented:", len(vehicle_cluster_counts))
print("Vehicles contributing to both clusters:", vehicles_both)
print("Vehicles contributing to only one cluster:", vehicles_one)

print("\nPercentage of vehicles contributing to both clusters:")
print(round(100 * vehicles_both / len(vehicle_cluster_counts), 2), "%")

# Examine the dominant cluster share for each vehicle
vehicle_cluster_counts["cluster_0_share"] = (
    vehicle_cluster_counts[0]
    / vehicle_cluster_counts[[0, 1]].sum(axis=1)
)

vehicle_cluster_counts["dominant_cluster_share"] = (
    vehicle_cluster_counts[["cluster_0_share"]]
    .assign(cluster_1_share=1 - vehicle_cluster_counts["cluster_0_share"])
    .max(axis=1)
)

print("\nDistribution of each vehicle's dominant-cluster share:")
display(
    vehicle_cluster_counts["dominant_cluster_share"]
    .describe()
    .round(3)
)

print("\nExample vehicles with the strongest cluster preference:")
display(
    vehicle_cluster_counts
    .sort_values("dominant_cluster_share", ascending=False)
    .head(10)
    [[0, 1, "cluster_0_share", "dominant_cluster_share"]]
    .round(3)
)


Vehicles represented: 150
Vehicles contributing to both clusters: 147
Vehicles contributing to only one cluster: 3

Percentage of vehicles contributing to both clusters:
98.0 %

Distribution of each vehicle's dominant-cluster share:


count    150.000
mean       0.674
std        0.116
min        0.507
25%        0.577
50%        0.660
75%        0.737
max        1.000
Name: dominant_cluster_share, dtype: float64


Example vehicles with the strongest cluster preference:


operational_2pc_k2,0,1,cluster_0_share,dominant_cluster_share
vehicle_id,,,,
64,255,0,1.000,1.000
115,1,0,1.000,1.000
65,241,0,1.000,1.000
19,1345,14,0.990,0.990
17,24,350,0.064,0.936
97,161,17,0.904,0.904
8,1,9,0.100,0.900
92,149,18,0.892,0.892
156,24,176,0.120,0.880


In [13]:

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
import pandas as pd
import numpy as np
import time

# Load the same two-PC matrix used in the baseline
X_stability_df = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pca_clustering_inputs"
    / "Operational_2PC.parquet"
)

# Reference labels from the original baseline run
reference_labels = labels_df["Operational_2PC_k2"].to_numpy()
X_stability = X_stability_df.to_numpy()

assert X_stability.shape[0] == len(reference_labels)
assert X_stability_df.index.equals(labels_df.index)

seeds = [0, 1, 7, 21, 42, 100, 2026]
stability_results = []

for seed in seeds:
    model = KMeans(
        n_clusters=2,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=seed,
        algorithm="lloyd",
    )

    start = time.perf_counter()
    labels = model.fit_predict(X_stability)
    runtime = time.perf_counter() - start

    ari = adjusted_rand_score(reference_labels, labels)

    stability_results.append({
        "seed": seed,
        "ARI_vs_baseline": ari,
        "inertia": model.inertia_,
        "iterations": model.n_iter_,
        "runtime_seconds": runtime,
    })

stability_df = pd.DataFrame(stability_results)

display(stability_df.round(4))

print(
    "\nMinimum ARI:",
    round(stability_df["ARI_vs_baseline"].min(), 4),
)
print(
    "Mean ARI:",
    round(stability_df["ARI_vs_baseline"].mean(), 4),
)


,seed,ARI_vs_baseline,inertia,iterations,runtime_seconds
0,0,0.9970,70541.1521,3,0.0354
1,1,0.9994,70541.6814,3,0.0329
2,7,0.9996,70541.7258,7,0.0291
3,21,1.0000,70541.8988,9,0.0281
4,42,1.0000,70541.8988,7,0.0286
5,100,0.9999,70541.8387,8,0.0302
6,2026,0.9995,70542.2225,8,0.0299



Minimum ARI: 0.997
Mean ARI: 0.9994


In [14]:

# Investigate whether tour_id can support trip-level aggregation.
# Start with the full engineered dataset, not only the PCA subset.

tour_data = tracks_df.copy()

print("Total track records:", len(tour_data))
print("Missing tour_id:", tour_data["tour_id"].isna().sum())
print("Unique tour_id values:", tour_data["tour_id"].nunique())
print("Unique vehicle_id values:", tour_data["vehicle_id"].nunique())

# Check whether a tour_id is associated with multiple vehicles.
tour_vehicle_counts = (
    tour_data.dropna(subset=["tour_id"])
    .groupby("tour_id")["vehicle_id"]
    .nunique()
)

print("\nTour IDs associated with multiple vehicles:")
print((tour_vehicle_counts > 1).sum())

# Use vehicle_id + tour_id as a provisional grouping key.
valid_tours = tour_data.dropna(
    subset=["vehicle_id", "tour_id"]
).copy()

tour_sizes = (
    valid_tours
    .groupby(["vehicle_id", "tour_id"])
    .size()
)

print("\nVehicle-tour groups:", len(tour_sizes))
print("Groups containing one track:", (tour_sizes == 1).sum())
print("Groups containing multiple tracks:", (tour_sizes > 1).sum())

print("\nTracks per vehicle-tour group:")
display(tour_sizes.describe().to_frame("tracks_per_group").round(2))

print("\nExample vehicle-tour groups:")
display(
    valid_tours.groupby(["vehicle_id", "tour_id"])
    .agg(
        track_count=("track_id", "size"),
        total_recorded_distance_km=("distance_km", "sum"),
        first_start=("start_time_parsed", "min"),
        last_stop=("stop_time_parsed", "max"),
    )
    .sort_values("track_count", ascending=False)
    .head(10)
)


Total track records: 151598
Missing tour_id: 0
Unique tour_id values: 13444
Unique vehicle_id values: 163

Tour IDs associated with multiple vehicles:
88

Vehicle-tour groups: 13579
Groups containing one track: 1533
Groups containing multiple tracks: 12046

Tracks per vehicle-tour group:


,tracks_per_group
count,13579.00
mean,11.16
std,36.70
min,1.00
25%,2.00
50%,4.00
75%,8.00
max,2138.00



Example vehicle-tour groups:


,,track_count,total_recorded_distance_km,first_start,last_stop
vehicle_id,tour_id,,,,
17,2862,2138,159.438,2021-12-20 13:55:17.077000+00:00,2022-01-18 08:22:12.399000+00:00
39,6916,910,39.180,2022-01-07 16:48:21.234000+00:00,2022-02-15 09:29:36.006000+00:00
57,10114,810,319.617,2022-01-07 17:42:50.404000+00:00,2022-01-26 09:07:18.004000+00:00
159,13428,706,20355.680,2023-09-14 11:59:35.545000+00:00,2024-02-05 12:55:20.625000+00:00
35,6641,663,191.559,2021-12-20 17:57:49.230000+00:00,2022-01-14 18:17:19.410000+00:00
31,6078,656,49.464,2021-12-23 11:43:40.051000+00:00,2022-01-14 06:53:07.748000+00:00
7,1299,634,65.684,2021-12-22 13:17:23.698000+00:00,2022-01-20 05:25:23.508000+00:00
1,39,588,118.251,2021-11-26 14:34:51.911000+00:00,2021-12-14 09:27:20.859000+00:00
56,10060,573,1823.047,2022-03-21 19:37:17.734000+00:00,2022-04-20 08:09:56.849000+00:00


In [16]:

# Validate the temporal continuity of each provisional vehicle-tour group

tour_tracks = (
    tracks_df
    .dropna(subset=["vehicle_id", "tour_id"])
    .copy()
)

# Explicitly parse timestamps; invalid values become NaT
tour_tracks["start_time_parsed"] = pd.to_datetime(
    tour_tracks["start_time_parsed"],
    errors="coerce",
    utc=True,
)

tour_tracks["stop_time_parsed"] = pd.to_datetime(
    tour_tracks["stop_time_parsed"],
    errors="coerce",
    utc=True,
)

print("Missing/invalid start timestamps:",
      tour_tracks["start_time_parsed"].isna().sum())
print("Missing/invalid stop timestamps:",
      tour_tracks["stop_time_parsed"].isna().sum())

# A valid start timestamp is required for temporal ordering
tour_tracks = (
    tour_tracks
    .dropna(subset=["start_time_parsed"])
    .sort_values(
        ["vehicle_id", "tour_id", "start_time_parsed"]
    )
)

group_cols = ["vehicle_id", "tour_id"]

# Summarize each vehicle-tour group
tour_summary = (
    tour_tracks
    .groupby(group_cols)
    .agg(
        track_count=("track_id", "size"),
        first_start=("start_time_parsed", "min"),
        last_start=("start_time_parsed", "max"),
        first_stop=("stop_time_parsed", "min"),
        last_stop=("stop_time_parsed", "max"),
        total_recorded_distance_km=("distance_km", "sum"),
    )
)

tour_summary["span_days"] = (
    tour_summary["last_start"] - tour_summary["first_start"]
).dt.total_seconds() / 86400

# Calculate gaps between consecutive track start times
tour_tracks["gap_hours_between_starts"] = (
    tour_tracks
    .groupby(group_cols)["start_time_parsed"]
    .diff()
    .dt.total_seconds() / 3600
)

gap_summary = (
    tour_tracks
    .groupby(group_cols)["gap_hours_between_starts"]
    .agg(
        median_gap_hours="median",
        max_gap_hours="max",
        gaps_over_6h=lambda s: (s > 6).sum(),
        gaps_over_24h=lambda s: (s > 24).sum(),
    )
)

tour_summary = tour_summary.join(gap_summary)

print("\nVehicle-tour groups:", len(tour_summary))

print("\nTemporal span distribution (days):")
display(
    tour_summary["span_days"]
    .describe()
    .round(2)
    .to_frame()
)

print("\nMaximum-gap distribution (hours):")
display(
    tour_summary["max_gap_hours"]
    .describe()
    .round(2)
    .to_frame()
)

print("\nGroups spanning more than 1 day:",
      (tour_summary["span_days"] > 1).sum())

print("Groups spanning more than 7 days:",
      (tour_summary["span_days"] > 7).sum())

print("Groups with a gap over 24 hours:",
      (tour_summary["gaps_over_24h"] > 0).sum())

print("\nTen groups with the longest temporal spans:")
display(
    tour_summary
    .sort_values("span_days", ascending=False)
    .head(10)
    .round(2)
)


Missing/invalid start timestamps: 145
Missing/invalid stop timestamps: 197

Vehicle-tour groups: 13577

Temporal span distribution (days):


,span_days
count,13577.00
mean,1.75
std,12.46
min,0.00
25%,0.04
50%,0.14
75%,0.49
max,234.57



Maximum-gap distribution (hours):


,max_gap_hours
count,12040.00
mean,25.11
std,203.64
min,0.05
25%,1.00
50%,2.16
75%,6.10
max,4123.99



Groups spanning more than 1 day: 1972
Groups spanning more than 7 days: 321
Groups with a gap over 24 hours: 671

Ten groups with the longest temporal spans:


/var/folders/3y/n0_t0qxd0r138yg74b00j8rh0000gn/T/ipykernel_1921/3631303928.py:109: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,,track_count,first_start,last_start,first_stop,last_stop,total_recorded_distance_km,span_days,median_gap_hours,max_gap_hours,gaps_over_6h,gaps_over_24h
vehicle_id,tour_id,,,,,,,,,,,
128,13127,313,2023-06-02 15:08:30.100000+00:00,2024-01-23 04:45:34.378000+00:00,2023-06-02 15:09:10.178000+00:00,2024-01-23 04:45:43.878000+00:00,7497.53,234.57,0.76,3181.65,40,16
134,13184,291,2023-06-02 15:21:06.476000+00:00,2024-01-19 05:39:07.346000+00:00,2023-06-02 15:24:09.162000+00:00,2024-01-19 05:39:17.874000+00:00,6217.46,230.60,0.84,4122.43,38,11
147,13291,309,2023-06-02 15:42:47.486000+00:00,2024-01-11 04:24:30.440000+00:00,2023-06-02 15:43:11.725000+00:00,2024-01-11 04:52:55.573000+00:00,7369.35,222.53,1.12,2441.25,47,17
158,13427,288,2023-06-02 13:23:10.069000+00:00,2024-01-10 09:14:53.488000+00:00,2023-06-02 13:31:32.484000+00:00,2024-01-10 09:15:02.885000+00:00,8669.09,221.83,1.06,1845.53,38,14
121,13080,295,2023-06-23 13:40:18.044000+00:00,2024-01-30 15:37:20.418000+00:00,2023-06-23 13:48:35.522000+00:00,2024-01-30 15:37:30.878000+00:00,5429.98,221.08,1.18,2655.08,62,16
153,13389,205,2023-06-02 15:48:45.810000+00:00,2024-01-09 12:41:33.859000+00:00,2023-06-02 15:51:59.954000+00:00,2024-01-09 12:46:15.545000+00:00,4326.54,220.87,0.84,2438.85,28,11
72,12860,303,2023-06-23 14:55:23.048000+00:00,2024-01-30 07:13:27.172000+00:00,2023-06-23 15:02:48.588000+00:00,2024-01-30 07:13:37.874000+00:00,3738.37,220.68,0.60,2653.46,41,16
90,12898,152,2023-06-23 14:14:16.111000+00:00,2024-01-29 12:12:18.865000+00:00,2023-06-23 14:24:49.220000+00:00,2024-01-29 13:12:05.615000+00:00,4899.09,219.92,3.85,2831.51,63,19
76,12876,296,2023-06-23 14:46:22.184000+00:00,2024-01-29 07:12:40.866000+00:00,2023-06-23 14:54:27.449000+00:00,2024-01-29 07:12:48.743000+00:00,10371.88,219.68,1.26,3479.96,47,13


In [17]:

# Inspect positive gaps between consecutive track starts.
# tour_tracks was already sorted by vehicle, tour, and start time.

gap_hours = tour_tracks["gap_hours_between_starts"].dropna()
gap_hours = gap_hours[gap_hours >= 0]

thresholds_hours = [1, 2, 4, 6, 8, 12, 24]

print("Positive gap distribution (hours):")
display(
    gap_hours.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2).to_frame("gap_hours")
)

threshold_summary = pd.DataFrame({
    "threshold_hours": thresholds_hours,
    "gaps_exceeding_threshold": [
        int((gap_hours > threshold).sum())
        for threshold in thresholds_hours
    ],
})

threshold_summary["percentage_of_gaps"] = (
    threshold_summary["gaps_exceeding_threshold"]
    / len(gap_hours) * 100
).round(2)

display(threshold_summary)

# Inspect the largest gaps with their associated group and timestamps.
largest_gaps = (
    tour_tracks.loc[
        tour_tracks["gap_hours_between_starts"].notna(),
        [
            "vehicle_id",
            "tour_id",
            "track_id",
            "start_time_parsed",
            "gap_hours_between_starts",
        ],
    ]
    .sort_values("gap_hours_between_starts", ascending=False)
    .head(15)
)

display(largest_gaps)


Positive gap distribution (hours):


,gap_hours
count,137876.00
mean,4.13
std,62.49
min,0.01
25%,0.30
50%,0.64
75%,1.46
90%,3.64
95%,8.36
99%,38.75


,threshold_hours,gaps_exceeding_threshold,percentage_of_gaps
0,1,48595,35.25
1,2,25320,18.36
2,4,12651,9.18
3,6,8862,6.43
4,8,7155,5.19
5,12,4997,3.62
6,24,1731,1.26


,vehicle_id,tour_id,track_id,start_time_parsed,gap_hours_between_starts
145731,144,13255,145731,2023-11-21 11:32:38.213000+00:00,4123.990582
142976,134,13184,142976,2023-11-21 09:47:10.032000+00:00,4122.434321
129204,76,12876,129204,2023-11-15 14:43:56.550000+00:00,3479.959546
141039,128,13127,141039,2023-10-13 05:04:59.192000+00:00,3181.645582
144804,140,13219,144804,2023-10-11 13:26:02.409000+00:00,3142.118580
129806,78,12876,129806,2023-10-24 12:55:34.484000+00:00,2950.150953
132281,90,12898,132281,2023-10-19 13:55:19.498000+00:00,2831.508302
137526,112,13006,137526,2023-10-19 04:39:08.052000+00:00,2823.308872
151009,162,13436,151009,2023-09-27 07:37:52.953000+00:00,2800.722113
135008,102,12981,135008,2023-09-26 09:42:27.710000+00:00,2777.805382


In [18]:

# Sort records chronologically within each vehicle-tour group.
segment_data = (
    tracks_df
    .dropna(subset=["vehicle_id", "tour_id"])
    .copy()
)

segment_data["start_time_parsed"] = pd.to_datetime(
    segment_data["start_time_parsed"],
    errors="coerce",
    utc=True,
)

segment_data["stop_time_parsed"] = pd.to_datetime(
    segment_data["stop_time_parsed"],
    errors="coerce",
    utc=True,
)

segment_data = (
    segment_data
    .dropna(subset=["start_time_parsed"])
    .sort_values(
        ["vehicle_id", "tour_id", "start_time_parsed"]
    )
)

group_cols = ["vehicle_id", "tour_id"]

# Previous track's stop time within the same vehicle-tour group.
segment_data["previous_stop"] = (
    segment_data
    .groupby(group_cols)["stop_time_parsed"]
    .shift(1)
)

# Actual gap between previous stop and current start.
segment_data["gap_after_previous_hours"] = (
    segment_data["start_time_parsed"]
    - segment_data["previous_stop"]
).dt.total_seconds() / 3600

valid_gaps = segment_data["gap_after_previous_hours"].dropna()

print("Gap summary (hours):")
display(
    valid_gaps.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).round(2).to_frame("gap_after_previous_hours")
)

print("\nNegative gaps (overlapping timestamps):",
      int((valid_gaps < 0).sum()))
print("Gaps over 24 hours:",
      int((valid_gaps > 24).sum()))

# Each group starts with one segment.
# A new segment begins when the gap exceeds the threshold.
thresholds_hours = [1, 2, 4, 6, 8, 12, 24]
group_count = segment_data.groupby(group_cols, sort=False).ngroups

threshold_results = []

for threshold in thresholds_hours:
    split_here = (
        segment_data["gap_after_previous_hours"] > threshold
    )

    # The first record in each group starts a new segment.
    first_in_group = (
        segment_data.groupby(group_cols, sort=False).cumcount() == 0
    )

    new_segment = split_here.fillna(False) | first_in_group

    # Count the resulting segments within each vehicle-tour group.
    segments_per_group = new_segment.groupby(
        [segment_data["vehicle_id"], segment_data["tour_id"]]
    ).sum()

    threshold_results.append({
        "threshold_hours": threshold,
        "provisional_segments": int(new_segment.sum()),
        "groups_with_multiple_segments": int(
            (segments_per_group > 1).sum()
        ),
        "groups_with_one_segment": int(
            (segments_per_group == 1).sum()
        ),
        "median_segments_per_group": float(
            segments_per_group.median()
        ),
    })

threshold_results_df = pd.DataFrame(threshold_results)
display(threshold_results_df.round(2))


Gap summary (hours):


,gap_after_previous_hours
count,137696.00
mean,3.78
std,62.53
min,0.00
25%,0.15
50%,0.35
75%,0.88
90%,2.80
95%,8.01
99%,38.13



Negative gaps (overlapping timestamps): 0
Gaps over 24 hours: 1714


,threshold_hours,provisional_segments,groups_with_multiple_segments,groups_with_one_segment,median_segments_per_group
0,1,44351,5770,7807,1.0
1,2,31269,4403,9174,1.0
2,4,24398,3438,10139,1.0
3,6,21994,2914,10663,1.0
4,8,20469,2410,11167,1.0
5,12,18166,1480,12097,1.0
6,24,15291,668,12909,1.0


In [19]:

# Inspect long_haul consistency within each vehicle-tour group.

long_haul_data = (
    tracks_df
    .dropna(subset=["vehicle_id", "tour_id"])
    .copy()
)

print("Overall long_haul distribution:")
display(
    long_haul_data["long_haul"]
    .value_counts(dropna=False)
    .to_frame("track_count")
)

print("\nLong-haul flag missing:", long_haul_data["long_haul"].isna().sum())

# Number of distinct non-missing long_haul values per group
tour_long_haul = (
    long_haul_data
    .groupby(["vehicle_id", "tour_id"])
    .agg(
        track_count=("track_id", "size"),
        distinct_long_haul_values=("long_haul", "nunique"),
        long_haul_true_share=(
            "long_haul",
            lambda s: (s == True).mean()
        ),
    )
)

print("\nVehicle-tour group consistency:")
display(
    tour_long_haul["distinct_long_haul_values"]
    .value_counts()
    .sort_index()
    .to_frame("group_count")
)

print("\nGroups with both True and False:")
print((tour_long_haul["distinct_long_haul_values"] > 1).sum())

print("\nLong-haul share distribution across groups:")
display(
    tour_long_haul["long_haul_true_share"]
    .describe()
    .round(3)
    .to_frame("long_haul_true_share")
)


Overall long_haul distribution:


,track_count
long_haul,
False,135159
True,16439



Long-haul flag missing: 0

Vehicle-tour group consistency:


,group_count
distinct_long_haul_values,
1,12411
2,1168



Groups with both True and False:
1168

Long-haul share distribution across groups:


,long_haul_true_share
count,13579.000
mean,0.047
std,0.171
min,0.000
25%,0.000
50%,0.000
75%,0.000
max,1.000


In [20]:

# Compare actual time gaps by the current track's long_haul flag.

gap_analysis = segment_data.loc[
    segment_data["gap_after_previous_hours"].notna()
    & segment_data["long_haul"].notna()
].copy()

gap_by_long_haul = (
    gap_analysis
    .groupby("long_haul")["gap_after_previous_hours"]
    .agg(
        count="count",
        median="median",
        mean="mean",
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
        over_4h=lambda s: (s > 4).mean() * 100,
        over_12h=lambda s: (s > 12).mean() * 100,
        over_24h=lambda s: (s > 24).mean() * 100,
    )
    .round(2)
)

display(gap_by_long_haul)


,count,median,mean,p75,p90,p95,over_4h,over_12h,over_24h
long_haul,,,,,,,,,
False,121871,0.35,3.99,0.89,2.67,7.55,7.49,3.37,1.3
True,15825,0.28,2.20,0.81,4.47,9.55,10.69,3.08,0.8


In [21]:

# Compare the previous and current track's long_haul labels.

transition_data = segment_data.copy()

transition_data["previous_long_haul"] = (
    transition_data
    .groupby(["vehicle_id", "tour_id"])["long_haul"]
    .shift(1)
)

transition_data = transition_data.dropna(
    subset=[
        "previous_long_haul",
        "long_haul",
        "gap_after_previous_hours",
    ]
)

transition_data["long_haul_transition"] = (
    transition_data["previous_long_haul"].astype(bool).map({
        False: "False"
    })
    + " -> "
    + transition_data["long_haul"].astype(bool).map({
        False: "False",
        True: "True",
    })
)

# Correctly label all four possible transitions.
transition_data["long_haul_transition"] = (
    transition_data["previous_long_haul"].map({False: "False", True: "True"})
    + " -> "
    + transition_data["long_haul"].map({False: "False", True: "True"})
)

transition_summary = (
    transition_data
    .groupby("long_haul_transition")["gap_after_previous_hours"]
    .agg(
        count="count",
        median="median",
        p90=lambda s: s.quantile(0.90),
        p95=lambda s: s.quantile(0.95),
        over_4h=lambda s: (s > 4).mean() * 100,
        over_12h=lambda s: (s > 12).mean() * 100,
    )
    .round(2)
)

display(transition_summary)


,count,median,p90,p95,over_4h,over_12h
long_haul_transition,,,,,,
False -> False,120099,0.35,2.60,7.33,7.36,3.29
False -> True,1195,0.34,12.54,16.96,17.57,10.54
True -> False,1772,0.53,10.21,16.06,16.42,8.18
True -> True,14630,0.27,4.05,9.07,10.13,2.47


In [22]:

# Compare fixed thresholds with a transition-aware candidate.
# This is a sensitivity experiment, not a final trip definition.

comparison = transition_data.copy()

comparison["is_transition"] = (
    comparison["previous_long_haul"] != comparison["long_haul"]
)

rules = {
    "Fixed_4h": lambda df: df["gap_after_previous_hours"] > 4,
    "Fixed_6h": lambda df: df["gap_after_previous_hours"] > 6,
    # Candidate: split ordinary transitions at 4h;
    # for flag transitions, report only gaps exceeding 2h for investigation.
    "Transition_review_2h": lambda df: (
        (~df["is_transition"] & (df["gap_after_previous_hours"] > 4))
        | (df["is_transition"] & (df["gap_after_previous_hours"] > 2))
    ),
}

rows = []

for rule_name, rule in rules.items():
    split_mask = rule(comparison)

    rows.append({
        "rule": rule_name,
        "gaps_evaluated": len(comparison),
        "gaps_marked": int(split_mask.sum()),
        "percentage_marked": round(100 * split_mask.mean(), 2),
        "marked_at_flag_transitions": int(
            (split_mask & comparison["is_transition"]).sum()
        ),
    })

display(pd.DataFrame(rows))


,rule,gaps_evaluated,gaps_marked,percentage_marked,marked_at_flag_transitions
0,Fixed_4h,137696,10821,7.86,501
1,Fixed_6h,137696,8417,6.11,417
2,Transition_review_2h,137696,11007,7.99,687


In [23]:

from pathlib import Path
import pandas as pd

# Project root directory
PROJECT_ROOT = Path.home() / "Documents" / "ML-Project"

# Dataset path
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "engineered_tracks.csv"

# Verify that the file exists before loading
print("Dataset path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

if DATA_PATH.exists():
    tracks_df = pd.read_csv(DATA_PATH)
    print("\nDataset loaded successfully.")
    print("Shape:", tracks_df.shape)
    display(tracks_df.head())
else:
    print("\nFile not found. Check the project directory and filename.")


Dataset path: /Users/janibhargesh/Documents/ML-Project/data/processed/engineered_tracks.csv
File exists: True

Dataset loaded successfully.
Shape: (151598, 63)


,track_id,vehicle_id,tour_id,start_time,stop_time,distance,track_gap,avg_speed,max_speed,n_signal_loss,...,n_signal_loss_valid,r_signal_loss_valid,distance_km,avg_speed_kmh,max_speed_kmh,raw_speed_mean_kmh,duration_minutes,avg_speed_implausible,speed_distance_validity,signal_loss_validity
0,0,1,0,2021-09-10 11:04:53.598000+02:00,2021-09-10 12:07:57.395000+02:00,30751.0,NaN,8.13,20.58,0.0,...,0.0,0.0,30.751,29.268,74.088,28.834092,63.063283,False,valid,valid
1,1,1,0,2021-09-10 12:57:54.718000+02:00,2021-09-10 13:13:53.221000+02:00,7762.0,37.504057,8.10,21.79,0.0,...,0.0,0.0,7.762,29.160,78.444,28.411624,15.975050,False,valid,valid
2,2,1,0,2021-09-10 13:25:20.146000+02:00,2021-09-10 14:09:50.802000+02:00,19432.0,4.895394,7.28,24.75,0.0,...,0.0,0.0,19.432,26.208,89.100,25.709182,44.510933,False,valid,valid
3,3,1,1,2021-09-10 14:21:35.547000+02:00,2021-09-10 14:25:26.391000+02:00,56.0,3.836167,0.24,2.44,0.0,...,0.0,0.0,0.056,0.864,8.784,0.456291,3.847400,False,valid,valid
4,4,1,1,2021-09-10 15:39:48.925000+02:00,2021-09-10 15:45:05.392000+02:00,201.0,12.816326,0.64,3.80,0.0,...,0.0,0.0,0.201,2.304,13.680,1.745397,5.274450,False,valid,valid


In [24]:

import pandas as pd
import numpy as np

# Work on a copy; do not modify the original dataset.
tour_check = tracks_df.copy()

# Parse timestamps safely, including timestamps stored as strings.
tour_check["start_time_parsed"] = pd.to_datetime(
    tour_check["start_time_parsed"],
    errors="coerce",
    utc=True
)

tour_check["stop_time_parsed"] = pd.to_datetime(
    tour_check["stop_time_parsed"],
    errors="coerce",
    utc=True
)

# Inspect required columns.
required_cols = [
    "vehicle_id",
    "tour_id",
    "track_id",
    "start_time_parsed",
    "stop_time_parsed",
    "home_base",
]

missing_cols = [
    col for col in required_cols
    if col not in tour_check.columns
]

print("Missing required columns:", missing_cols)

if not missing_cols:
    print("\nDataset shape:", tour_check.shape)
    print("Unique vehicles:", tour_check["vehicle_id"].nunique())
    print("Unique tour IDs:", tour_check["tour_id"].nunique())
    print(
        "Unique vehicle-tour pairs:",
        tour_check[["vehicle_id", "tour_id"]]
        .drop_duplicates()
        .shape[0]
    )

    print("\nMissing values in key fields:")
    print(
        tour_check[
            ["vehicle_id", "tour_id", "track_id",
             "start_time_parsed", "stop_time_parsed", "home_base"]
        ].isna().sum()
    )


Missing required columns: []

Dataset shape: (151598, 63)
Unique vehicles: 163
Unique tour IDs: 13444
Unique vehicle-tour pairs: 13579

Missing values in key fields:
vehicle_id             0
tour_id                0
track_id               0
start_time_parsed    145
stop_time_parsed     197
home_base              0
dtype: int64


In [25]:

# Step 2: Validate vehicle-tour groups chronologically

# Use only rows with valid start times for chronological checks.
tour_time_data = tour_check.dropna(
    subset=["vehicle_id", "tour_id", "start_time_parsed"]
).copy()

tour_time_data = tour_time_data.sort_values(
    ["vehicle_id", "tour_id", "start_time_parsed"]
)

group_cols = ["vehicle_id", "tour_id"]

# Summarize each vehicle-tour group.
tour_summary = (
    tour_time_data.groupby(group_cols)
    .agg(
        track_count=("track_id", "nunique"),
        first_start=("start_time_parsed", "min"),
        last_start=("start_time_parsed", "max"),
        home_base_true_count=("home_base", "sum"),
        home_base_value_count=("home_base", "count"),
    )
    .reset_index()
)

tour_summary["span_days"] = (
    tour_summary["last_start"] - tour_summary["first_start"]
).dt.total_seconds() / 86400

print("Vehicle-tour groups:", len(tour_summary))
print("\nTrack count per group:")
print(tour_summary["track_count"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nGroup span in days:")
print(tour_summary["span_days"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nGroups spanning more than:")
for threshold in [1, 3, 7, 30, 90]:
    count = (tour_summary["span_days"] > threshold).sum()
    print(f"{threshold:>2} days: {count:,}")

# Inspect the longest groups.
print("\n20 longest vehicle-tour groups:")
display(
    tour_summary.sort_values("span_days", ascending=False).head(20)
)


Vehicle-tour groups: 13577

Track count per group:
count    13577.000000
mean        11.155115
std         36.666074
min          1.000000
50%          4.000000
75%          8.000000
90%         19.000000
95%         40.000000
99%        132.000000
max       2137.000000
Name: track_count, dtype: float64

Group span in days:
count    13577.000000
mean         1.746404
std         12.458379
min          0.000000
50%          0.140100
75%          0.486340
90%          2.011338
95%          3.373761
99%         29.323972
max        234.567411
Name: span_days, dtype: float64

Groups spanning more than:
 1 days: 1,972
 3 days: 761
 7 days: 321
30 days: 135
90 days: 81

20 longest vehicle-tour groups:


,vehicle_id,tour_id,track_count,first_start,last_start,home_base_true_count,home_base_value_count,span_days
13225,128,13127,313,2023-06-02 15:08:30.100000+00:00,2024-01-23 04:45:34.378000+00:00,0,313,234.567411
13288,134,13184,291,2023-06-02 15:21:06.476000+00:00,2024-01-19 05:39:07.346000+00:00,0,291,230.595843
13408,147,13291,309,2023-06-02 15:42:47.486000+00:00,2024-01-11 04:24:30.440000+00:00,0,309,222.528969
13555,158,13427,288,2023-06-02 13:23:10.069000+00:00,2024-01-10 09:14:53.488000+00:00,0,288,221.827586
13171,121,13080,295,2023-06-23 13:40:18.044000+00:00,2024-01-30 15:37:20.418000+00:00,0,295,221.081277
13512,153,13389,205,2023-06-02 15:48:45.810000+00:00,2024-01-09 12:41:33.859000+00:00,0,205,220.870001
12905,72,12860,303,2023-06-23 14:55:23.048000+00:00,2024-01-30 07:13:27.172000+00:00,0,303,220.679214
12961,90,12898,152,2023-06-23 14:14:16.111000+00:00,2024-01-29 12:12:18.865000+00:00,0,152,219.915310
12925,76,12876,296,2023-06-23 14:46:22.184000+00:00,2024-01-29 07:12:40.866000+00:00,0,296,219.684938
12962,91,12898,284,2023-06-23 14:14:05.051000+00:00,2024-01-26 13:16:54.806000+00:00,0,284,216.960298


In [26]:

# Step 3: Inspect individual records from suspicious vehicle-tour groups

inspect_pairs = [
    (128, 13127),
    (134, 13184),
]

inspection_cols = [
    "vehicle_id",
    "tour_id",
    "track_id",
    "start_time_parsed",
    "stop_time_parsed",
    "home_base",
    "long_haul",
    "rest_area",
    "service_area_fuel",
    "industrial_area",
]

# Keep only columns that actually exist in the dataset.
inspection_cols = [
    col for col in inspection_cols if col in tour_check.columns
]

for vehicle, tour in inspect_pairs:
    group = tour_check.loc[
        (tour_check["vehicle_id"] == vehicle)
        & (tour_check["tour_id"] == tour)
    ].copy()

    group = group.sort_values("start_time_parsed")

    group["previous_stop"] = group["stop_time_parsed"].shift(1)
    group["gap_after_previous_hours"] = (
        group["start_time_parsed"] - group["previous_stop"]
    ).dt.total_seconds() / 3600

    print(f"\nVehicle {vehicle}, tour_id {tour}")
    print("Rows:", len(group))
    print(
        "Date range:",
        group["start_time_parsed"].min(),
        "to",
        group["start_time_parsed"].max()
    )

    # Show beginning and end of the group.
    print("\nFirst 10 tracks:")
    display(group[inspection_cols + [
        "previous_stop",
        "gap_after_previous_hours"
    ]].head(10))

    print("\nLast 10 tracks:")
    display(group[inspection_cols + [
        "previous_stop",
        "gap_after_previous_hours"
    ]].tail(10))

    print("\nLargest gaps between consecutive tracks:")
    display(
        group.loc[
            group["gap_after_previous_hours"] > 24,
            inspection_cols + [
                "previous_stop",
                "gap_after_previous_hours"
            ]
        ].sort_values(
            "gap_after_previous_hours",
            ascending=False
        ).head(10)
    )



Vehicle 128, tour_id 13127
Rows: 313
Date range: 2023-06-02 15:08:30.100000+00:00 to 2024-01-23 04:45:34.378000+00:00

First 10 tracks:


,vehicle_id,tour_id,track_id,start_time_parsed,stop_time_parsed,home_base,long_haul,rest_area,service_area_fuel,industrial_area,previous_stop,gap_after_previous_hours
141036,128,13127,141036,2023-06-02 15:08:30.100000+00:00,2023-06-02 15:09:10.178000+00:00,False,False,False,False,False,NaT,NaN
141037,128,13127,141037,2023-06-02 15:11:51.719000+00:00,2023-06-02 15:14:10.470000+00:00,False,False,False,False,False,2023-06-02 15:09:10.178000+00:00,0.044872
141038,128,13127,141038,2023-06-02 15:26:15.095000+00:00,2023-06-02 15:30:06.042000+00:00,False,False,False,False,False,2023-06-02 15:14:10.470000+00:00,0.201285
141039,128,13127,141039,2023-10-13 05:04:59.192000+00:00,2023-10-13 06:55:13.329000+00:00,False,False,False,False,False,2023-06-02 15:30:06.042000+00:00,3181.581431
141040,128,13127,141040,2023-10-13 07:54:18.130000+00:00,2023-10-13 08:08:38.566000+00:00,False,False,False,False,False,2023-10-13 06:55:13.329000+00:00,0.984667
141041,128,13127,141041,2023-10-13 09:00:29.998000+00:00,2023-10-13 09:07:35.042000+00:00,False,False,False,False,True,2023-10-13 08:08:38.566000+00:00,0.864287
141042,128,13127,141042,2023-10-13 09:11:51.186000+00:00,2023-10-13 09:18:31.692000+00:00,False,False,False,False,True,2023-10-13 09:07:35.042000+00:00,0.071151
141043,128,13127,141043,2023-10-13 09:21:25.583000+00:00,2023-10-13 09:53:35.689000+00:00,False,False,False,False,False,2023-10-13 09:18:31.692000+00:00,0.048303
141044,128,13127,141044,2023-10-16 04:37:11.625000+00:00,2023-10-16 05:37:10.612000+00:00,False,False,False,False,True,2023-10-13 09:53:35.689000+00:00,66.726649
141045,128,13127,141045,2023-10-16 06:45:57.222000+00:00,2023-10-16 06:53:18.631000+00:00,False,False,False,False,True,2023-10-16 05:37:10.612000+00:00,1.146281



Last 10 tracks:


,vehicle_id,tour_id,track_id,start_time_parsed,stop_time_parsed,home_base,long_haul,rest_area,service_area_fuel,industrial_area,previous_stop,gap_after_previous_hours
141339,128,13127,141339,2023-11-27 14:48:27.656000+00:00,2023-11-27 16:11:47.745000+00:00,False,False,False,False,False,2023-11-27 09:48:03.996000+00:00,5.006572
141340,128,13127,141340,2023-11-29 14:40:31.582000+00:00,2023-11-29 14:40:40.855000+00:00,False,False,False,False,False,2023-11-27 16:11:47.745000+00:00,46.478844
141341,128,13127,141341,2023-12-12 08:46:55.059000+00:00,2023-12-12 10:42:33.223000+00:00,False,False,False,False,False,2023-11-29 14:40:40.855000+00:00,306.103946
141342,128,13127,141342,2023-12-27 06:41:18.277000+00:00,2023-12-27 06:45:41.452000+00:00,False,False,False,False,False,2023-12-12 10:42:33.223000+00:00,355.979182
141343,128,13127,141343,2023-12-29 10:00:39.776000+00:00,2023-12-29 11:20:36.278000+00:00,False,False,False,False,True,2023-12-27 06:45:41.452000+00:00,51.249534
141344,128,13127,141344,2024-01-09 10:21:35.230000+00:00,2024-01-09 11:39:05.642000+00:00,False,False,False,False,True,2023-12-29 11:20:36.278000+00:00,263.016376
141345,128,13127,141345,2024-01-10 08:14:45.411000+00:00,2024-01-10 08:18:07.456000+00:00,False,False,False,False,True,2024-01-09 11:39:05.642000+00:00,20.594380
141346,128,13127,141346,2024-01-10 12:01:56.545000+00:00,2024-01-10 13:05:22.790000+00:00,False,False,False,False,False,2024-01-10 08:18:07.456000+00:00,3.730303
141347,128,13127,141347,2024-01-17 11:44:42.548000+00:00,2024-01-17 11:44:51.886000+00:00,False,False,False,False,False,2024-01-10 13:05:22.790000+00:00,166.655488
141348,128,13127,141348,2024-01-23 04:45:34.378000+00:00,2024-01-23 04:45:43.878000+00:00,False,False,False,False,False,2024-01-17 11:44:51.886000+00:00,137.011803



Largest gaps between consecutive tracks:


...

,vehicle_id,tour_id,track_id,start_time_parsed,stop_time_parsed,home_base,long_haul,rest_area,service_area_fuel,industrial_area,previous_stop,gap_after_previous_hours
142975,134,13184,142975,2023-06-02 15:21:06.476000+00:00,2023-06-02 15:24:09.162000+00:00,False,False,False,False,False,NaT,NaN
142976,134,13184,142976,2023-11-21 09:47:10.032000+00:00,2023-11-21 09:48:33.492000+00:00,False,False,False,False,False,2023-06-02 15:24:09.162000+00:00,4122.383575
142977,134,13184,142977,2023-11-21 09:56:54.871000+00:00,2023-11-21 09:57:35.468000+00:00,False,False,False,False,False,2023-11-21 09:48:33.492000+00:00,0.139272
142978,134,13184,142978,2023-11-21 10:23:56.080000+00:00,2023-11-21 11:50:36.033000+00:00,False,False,False,False,False,2023-11-21 09:57:35.468000+00:00,0.439059
142979,134,13184,142979,2023-11-21 11:55:33.830000+00:00,2023-11-21 12:07:08.805000+00:00,False,False,False,False,True,2023-11-21 11:50:36.033000+00:00,0.082721
142980,134,13184,142980,2023-11-21 13:45:12.347000+00:00,2023-11-21 15:41:21.775000+00:00,False,False,False,False,False,2023-11-21 12:07:08.805000+00:00,1.634317
142981,134,13184,142981,2023-11-21 15:45:02.058000+00:00,2023-11-21 16:07:31.400000+00:00,False,False,False,False,False,2023-11-21 15:41:21.775000+00:00,0.061190
142982,134,13184,142982,2023-11-22 05:09:17.699000+00:00,2023-11-22 06:48:47.383000+00:00,False,False,False,False,True,2023-11-21 16:07:31.400000+00:00,13.029528
142983,134,13184,142983,2023-11-22 06:52:18.425000+00:00,2023-11-22 06:53:31.742000+00:00,False,False,False,False,True,2023-11-22 06:48:47.383000+00:00,0.058623
142984,134,13184,142984,2023-11-22 06:58:47.231000+00:00,2023-11-22 07:01:05.985000+00:00,False,False,False,False,True,2023-11-22 06:53:31.742000+00:00,0.087636



Last 10 tracks:


,vehicle_id,tour_id,track_id,start_time_parsed,stop_time_parsed,home_base,long_haul,rest_area,service_area_fuel,industrial_area,previous_stop,gap_after_previous_hours
143256,134,13184,143256,2024-01-10 05:08:19.091000+00:00,2024-01-10 06:02:55.424000+00:00,False,False,False,False,False,2024-01-09 13:59:18.962000+00:00,15.150036
143257,134,13184,143257,2024-01-11 10:58:38.215000+00:00,2024-01-11 10:58:47.876000+00:00,False,False,False,False,False,2024-01-10 06:02:55.424000+00:00,28.928553
143258,134,13184,143258,2024-01-15 06:22:38.551000+00:00,2024-01-15 06:45:35.494000+00:00,False,False,False,False,True,2024-01-11 10:58:47.876000+00:00,91.397410
143259,134,13184,143259,2024-01-15 06:53:24.658000+00:00,2024-01-15 06:57:12.782000+00:00,False,False,False,False,True,2024-01-15 06:45:35.494000+00:00,0.130323
143260,134,13184,143260,2024-01-15 10:00:14.258000+00:00,2024-01-15 10:00:23.874000+00:00,False,False,False,False,False,2024-01-15 06:57:12.782000+00:00,3.050410
143261,134,13184,143261,2024-01-15 11:31:35.569000+00:00,2024-01-15 11:31:43.447000+00:00,False,False,False,False,True,2024-01-15 10:00:23.874000+00:00,1.519915
143262,134,13184,143262,2024-01-15 11:56:28.356000+00:00,2024-01-15 12:04:40.470000+00:00,False,False,False,False,False,2024-01-15 11:31:43.447000+00:00,0.412475
143263,134,13184,143263,2024-01-18 05:28:31.653000+00:00,2024-01-18 05:53:45.962000+00:00,False,False,False,False,False,2024-01-15 12:04:40.470000+00:00,65.397551
143264,134,13184,143264,2024-01-18 11:07:30.503000+00:00,2024-01-18 11:14:04.474000+00:00,False,False,False,False,False,2024-01-18 05:53:45.962000+00:00,5.229039
143265,134,13184,143265,2024-01-19 05:39:07.346000+00:00,2024-01-19 05:39:17.874000+00:00,False,False,False,False,False,2024-01-18 11:14:04.474000+00:00,18.417464



Largest gaps between consecutive tracks:


,vehicle_id,tour_id,track_id,start_time_parsed,stop_time_parsed,home_base,long_haul,rest_area,service_area_fuel,industrial_area,previous_stop,gap_after_previous_hours
142976,134,13184,142976,2023-11-21 09:47:10.032000+00:00,2023-11-21 09:48:33.492000+00:00,False,False,False,False,False,2023-06-02 15:24:09.162000+00:00,4122.383575
143208,134,13184,143208,2023-12-28 08:58:31.522000+00:00,2023-12-28 09:52:31.854000+00:00,False,False,False,False,False,2023-12-22 12:46:08.843000+00:00,140.206300
143224,134,13184,143224,2024-01-02 09:07:43.181000+00:00,2024-01-02 09:07:51.874000+00:00,False,False,False,False,True,2023-12-29 12:04:25.633000+00:00,93.054874
143258,134,13184,143258,2024-01-15 06:22:38.551000+00:00,2024-01-15 06:45:35.494000+00:00,False,False,False,False,True,2024-01-11 10:58:47.876000+00:00,91.397410
143046,134,13184,143046,2023-12-04 06:40:19.218000+00:00,2023-12-04 09:42:29.015000+00:00,False,False,False,False,True,2023-12-01 12:34:37.137000+00:00,66.095022
143263,134,13184,143263,2024-01-18 05:28:31.653000+00:00,2024-01-18 05:53:45.962000+00:00,False,False,False,False,False,2024-01-15 12:04:40.470000+00:00,65.397551
143251,134,13184,143251,2024-01-08 05:15:54.611000+00:00,2024-01-08 08:54:22.702000+00:00,False,False,False,False,True,2024-01-05 12:59:13.961000+00:00,64.277958
143102,134,13184,143102,2023-12-11 04:47:49.082000+00:00,2023-12-11 05:01:51.286000+00:00,False,False,False,False,False,2023-12-08 14:26:46.831000+00:00,62.350625
143166,134,13184,143166,2023-12-18 05:24:09.122000+00:00,2023-12-18 08:01:22.383000+00:00,False,False,False,False,False,2023-12-15 16:30:56.623000+00:00,60.886805
143012,134,13184,143012,2023-11-27 05:31:13.579000+00:00,2023-11-27 06:02:52.420000+00:00,False,False,False,False,True,2023-11-24 17:12:36.205000+00:00,60.310382


In [27]:

# Step 5: Check tour-ID transitions against home-base endpoints

boundary_check = tour_check.dropna(
    subset=["vehicle_id", "start_time_parsed"]
).copy()

boundary_check = boundary_check.sort_values(
    ["vehicle_id", "start_time_parsed"]
)

vehicle_group = boundary_check.groupby("vehicle_id", sort=False)

boundary_check["previous_tour_id"] = (
    vehicle_group["tour_id"].shift(1)
)

boundary_check["previous_home_base"] = (
    vehicle_group["home_base"].shift(1)
)

boundary_check["tour_id_changed"] = (
    boundary_check["tour_id"] != boundary_check["previous_tour_id"]
)

# A home-base endpoint on the previous track is the boundary
# we expect to examine before the next track begins.
boundary_check["previous_track_ended_home"] = (
    boundary_check["previous_home_base"].eq(True)
)

valid_transitions = boundary_check[
    boundary_check["previous_tour_id"].notna()
].copy()

print("Track transitions examined:", len(valid_transitions))

print("\nDid the tour ID change?")
print(valid_transitions["tour_id_changed"].value_counts())

print("\nTour-ID changes by previous track's home-base label:")
display(pd.crosstab(
    valid_transitions["previous_track_ended_home"],
    valid_transitions["tour_id_changed"],
    rownames=["Previous track ended at home base"],
    colnames=["Tour ID changed"]
))

print("\nExamples where ID stayed the same after a home-base endpoint:")
display(
    valid_transitions.loc[
        valid_transitions["previous_track_ended_home"]
        & ~valid_transitions["tour_id_changed"],
        [
            "vehicle_id", "previous_tour_id", "tour_id",
            "track_id", "start_time_parsed",
            "previous_home_base", "home_base"
        ]
    ].head(20)
)

print("\nExamples where ID changed without a previous home-base endpoint:")
display(
    valid_transitions.loc[
        ~valid_transitions["previous_track_ended_home"]
        & valid_transitions["tour_id_changed"],
        [
            "vehicle_id", "previous_tour_id", "tour_id",
            "track_id", "start_time_parsed",
            "previous_home_base", "home_base"
        ]
    ].head(20)
)


Track transitions examined: 151290

Did the tour ID change?
tour_id_changed
False    137876
True      13414
Name: count, dtype: int64

Tour-ID changes by previous track's home-base label:


Tour ID changed,False,True
Previous track ended at home base,,
False,75332,11
True,62544,13403



Examples where ID stayed the same after a home-base endpoint:


,vehicle_id,previous_tour_id,tour_id,track_id,start_time_parsed,previous_home_base,home_base
4,1,1.0,1,4,2021-09-10 13:39:48.925000+00:00,True,True
5,1,1.0,1,5,2021-09-11 01:12:11.008000+00:00,True,True
6,1,1.0,1,6,2021-09-13 04:23:58.663000+00:00,True,True
7,1,1.0,1,7,2021-09-13 04:59:58.115000+00:00,True,True
8,1,1.0,1,8,2021-09-13 05:58:54.717000+00:00,True,True
9,1,1.0,1,9,2021-09-13 06:08:10.018000+00:00,True,True
19,1,3.0,3,19,2021-09-15 06:30:49.367000+00:00,True,True
20,1,3.0,3,20,2021-09-15 08:17:33.911000+00:00,True,True
21,1,3.0,3,21,2021-09-16 09:46:12.359000+00:00,True,True
22,1,3.0,3,22,2021-09-16 11:47:23.895000+00:00,True,True



Examples where ID changed without a previous home-base endpoint:


,vehicle_id,previous_tour_id,tour_id,track_id,start_time_parsed,previous_home_base,home_base
3896,2,270.0,271,3896,2022-01-07 09:58:09.262000+00:00,False,True
15516,9,1349.0,1350,15516,2021-09-13 12:16:16.131000+00:00,False,True
15579,9,1359.0,1360,15579,2021-09-20 23:50:20.053000+00:00,False,True
66528,31,6037.0,6038,66528,2021-12-01 12:44:51.641000+00:00,False,True
109495,56,9809.0,9810,109495,2021-12-30 02:56:02.756000+00:00,False,True
109865,56,9859.0,9860,109865,2022-01-13 09:21:10.241000+00:00,False,True
115866,59,10880.0,10881,115866,2022-03-02 03:44:14.502000+00:00,False,True
117076,59,11268.0,11269,117076,2022-05-21 18:42:15.112000+00:00,False,False
117598,59,11444.0,11445,117598,2022-06-29 18:33:22.564000+00:00,False,True
127189,67,12847.0,12848,127189,2023-10-23 03:44:17.374000+00:00,False,False


In [28]:

# Step 6: Inspect fields that may explain tour-ID transitions

candidate_cols = [
    "vehicle_id",
    "track_id",
    "tour_id",
    "previous_tour_id",
    "start_time_parsed",
    "stop_time_parsed",
    "previous_stop",
    "gap_after_previous_hours",
    "previous_home_base",
    "home_base",
    "cid",
    "destination_cluster_id",
    "start_location_cluster_id",
    "end_location_cluster_id",
]

available_cols = [
    col for col in candidate_cols
    if col in sequence_df.columns
]

# Show every transition where the supplied ID changes.
id_changes = sequence_df.loc[
    sequence_df["tour_id_changed"],
    available_cols
].copy()

print("Available diagnostic columns:", available_cols)
print("Tour-ID changes:", len(id_changes))

display(id_changes.head(30))

print("\nChanges after a non-home-base endpoint:")
display(
    id_changes.loc[
        id_changes["previous_home_base"].eq(False)
    ]
)


NameError: name 'sequence_df' is not defined

In [29]:

# Step 6: Self-contained tour-ID transition diagnostic

import pandas as pd
import numpy as np

# Rebuild the chronological dataset from tour_check
sequence_df = tour_check.dropna(
    subset=["vehicle_id", "start_time_parsed"]
).copy()

sequence_df = sequence_df.sort_values(
    ["vehicle_id", "start_time_parsed"]
)

# Previous track's tour ID and home-base endpoint, per vehicle
vehicle_group = sequence_df.groupby("vehicle_id", sort=False)

sequence_df["previous_tour_id"] = (
    vehicle_group["tour_id"].shift(1)
)

sequence_df["previous_home_base"] = (
    vehicle_group["home_base"].shift(1)
)

sequence_df["previous_stop"] = (
    vehicle_group["stop_time_parsed"].shift(1)
)

sequence_df["gap_after_previous_hours"] = (
    sequence_df["start_time_parsed"] - sequence_df["previous_stop"]
).dt.total_seconds() / 3600

sequence_df["tour_id_changed"] = (
    sequence_df["tour_id"] != sequence_df["previous_tour_id"]
)

# Find which location-cluster fields are actually available
candidate_cols = [
    "vehicle_id", "track_id",
    "previous_tour_id", "tour_id",
    "start_time_parsed", "stop_time_parsed",
    "previous_stop", "gap_after_previous_hours",
    "previous_home_base", "home_base",
    "cid", "destination_cluster_id",
    "start_location_cluster_id", "end_location_cluster_id",
]

available_cols = [
    col for col in candidate_cols
    if col in sequence_df.columns
]

id_changes = sequence_df.loc[
    sequence_df["tour_id_changed"]
    & sequence_df["previous_tour_id"].notna(),
    available_cols
].copy()

print("Available diagnostic columns:", available_cols)
print("Tour-ID changes:", len(id_changes))

print("\nFirst 30 tour-ID changes:")
display(id_changes.head(30))

print("\nChanges after a non-home-base endpoint:")
display(
    id_changes.loc[
        id_changes["previous_home_base"].eq(False)
    ]
)


Available diagnostic columns: ['vehicle_id', 'track_id', 'previous_tour_id', 'tour_id', 'start_time_parsed', 'stop_time_parsed', 'previous_stop', 'gap_after_previous_hours', 'previous_home_base', 'home_base', 'cid']
Tour-ID changes: 13414

First 30 tour-ID changes:


,vehicle_id,track_id,previous_tour_id,tour_id,start_time_parsed,stop_time_parsed,previous_stop,gap_after_previous_hours,previous_home_base,home_base,cid
3,1,3,0.0,1,2021-09-10 12:21:35.547000+00:00,2021-09-10 12:25:26.391000+00:00,2021-09-10 12:09:50.802000+00:00,0.195763,True,True,0
10,1,10,1.0,2,2021-09-13 06:25:03.766000+00:00,2021-09-13 06:37:36.886000+00:00,2021-09-13 06:08:26.837000+00:00,0.276925,True,False,301
18,1,18,2.0,3,2021-09-14 11:49:23.603000+00:00,2021-09-14 12:13:47.314000+00:00,2021-09-14 11:45:39.906000+00:00,0.062138,True,True,0
122,1,122,3.0,4,2021-10-07 10:19:52.098000+00:00,2021-10-07 11:17:39.292000+00:00,2021-10-07 08:49:40.058000+00:00,1.503344,True,False,149
138,1,138,4.0,5,2021-10-09 08:07:14.706000+00:00,2021-10-09 08:07:57.210000+00:00,2021-10-09 07:59:24.414000+00:00,0.130637,True,True,0
186,1,186,5.0,6,2021-10-11 06:12:38.147000+00:00,2021-10-11 08:04:30.820000+00:00,2021-10-11 06:02:24.076000+00:00,0.170575,True,False,157
191,1,191,6.0,7,2021-10-11 11:39:45.708000+00:00,2021-10-11 11:43:02.289000+00:00,2021-10-11 11:27:17.033000+00:00,0.207965,True,True,0
192,1,192,7.0,8,2021-10-11 13:05:55.392000+00:00,2021-10-11 15:37:12.004000+00:00,2021-10-11 11:43:02.289000+00:00,1.381417,True,False,327
210,1,210,8.0,9,2021-10-13 15:14:26.905000+00:00,2021-10-13 15:36:42.103000+00:00,2021-10-13 13:45:49.873000+00:00,1.476953,True,True,0
211,1,211,9.0,10,2021-10-13 15:48:39.651000+00:00,2021-10-13 20:01:33.883000+00:00,2021-10-13 15:36:42.103000+00:00,0.199319,True,False,-1



Changes after a non-home-base endpoint:


,vehicle_id,track_id,previous_tour_id,tour_id,start_time_parsed,stop_time_parsed,previous_stop,gap_after_previous_hours,previous_home_base,home_base,cid
3896,2,3896,270.0,271,2022-01-07 09:58:09.262000+00:00,2022-01-07 10:01:08.332000+00:00,2022-01-07 07:47:23.928000+00:00,2.179259,False,True,0
15516,9,15516,1349.0,1350,2021-09-13 12:16:16.131000+00:00,2021-09-13 13:12:38.043000+00:00,2021-09-13 09:40:15.687000+00:00,2.600123,False,True,0
15579,9,15579,1359.0,1360,2021-09-20 23:50:20.053000+00:00,2021-09-20 23:50:36.875000+00:00,2021-09-20 12:51:00.878000+00:00,10.988660,False,True,0
66528,31,66528,6037.0,6038,2021-12-01 12:44:51.641000+00:00,2021-12-01 12:45:42.689000+00:00,2021-12-01 11:47:24.910000+00:00,0.957425,False,True,2
109495,56,109495,9809.0,9810,2021-12-30 02:56:02.756000+00:00,2021-12-30 02:58:53.699000+00:00,2021-12-30 00:44:01.458000+00:00,2.200361,False,True,102
109865,56,109865,9859.0,9860,2022-01-13 09:21:10.241000+00:00,2022-01-13 09:21:27.061000+00:00,2022-01-13 05:31:26.949000+00:00,3.828692,False,True,102
115866,59,115866,10880.0,10881,2022-03-02 03:44:14.502000+00:00,2022-03-02 03:44:31.325000+00:00,2022-03-02 01:29:15.071000+00:00,2.249842,False,True,102
117076,59,117076,11268.0,11269,2022-05-21 18:42:15.112000+00:00,2022-05-21 18:46:40.258000+00:00,2022-05-21 13:22:06.378000+00:00,5.335759,False,False,1164
117598,59,117598,11444.0,11445,2022-06-29 18:33:22.564000+00:00,2022-06-29 18:37:47.660000+00:00,2022-06-29 16:23:08.698000+00:00,2.170518,False,True,102
127189,67,127189,12847.0,12848,2023-10-23 03:44:17.374000+00:00,2023-10-23 05:00:15.841000+00:00,2023-10-18 13:05:46.468000+00:00,110.641918,False,False,809


In [30]:

# Step 7: Examine endpoint cluster IDs and home-base labels

print("Unique cid values:", tour_check["cid"].nunique())
print("\nMost common cid values:")
print(tour_check["cid"].value_counts(dropna=False).head(20))

print("\nHome-base label by cid:")
cid_home_summary = (
    tour_check.groupby("cid", dropna=False)
    .agg(
        track_count=("track_id", "count"),
        home_base_true_rate=("home_base", "mean"),
        unique_vehicles=("vehicle_id", "nunique"),
        unique_tour_ids=("tour_id", "nunique"),
    )
    .sort_values("track_count", ascending=False)
)

display(cid_home_summary.head(30))

print("\nHome-base label distribution:")
display(pd.crosstab(
    tour_check["cid"],
    tour_check["home_base"],
    margins=True
).head(30))


Unique cid values: 1308

Most common cid values:
cid
 0       26713
 502     12045
 745      9190
 809      8271
 102      7466
 26       6185
 2        5002
 412      3899
 1251     3817
-1        3501
 4        3216
 1252     2341
 20       2020
 514      1539
 1159     1519
 825      1477
 812      1419
 123      1258
 842      1096
 3        1060
Name: count, dtype: int64

Home-base label by cid:


,track_count,home_base_true_rate,unique_vehicles,unique_tour_ids
cid,,,,
0,26713,0.999326,19,2976
502,12045,0.980158,18,1654
745,9190,1.000000,5,1362
809,8271,0.080401,71,614
102,7466,0.998928,7,2648
26,6185,0.999838,12,1052
2,5002,0.999800,10,872
412,3899,0.000000,44,58
1251,3817,0.719937,4,887



Home-base label distribution:


home_base,False,True,All
cid,,,
-1,3501,0,3501
0,18,26695,26713
1,35,0,35
2,1,5001,5002
3,1060,0,1060
4,12,3204,3216
5,68,0,68
6,13,0,13
7,48,0,48


In [31]:
import pandas as pd

df = tracks_df.copy()

df["start_time_parsed"] = pd.to_datetime(
    df["start_time"], errors="coerce", utc=True
)
df["stop_time_parsed"] = pd.to_datetime(
    df["stop_time"], errors="coerce", utc=True
)

df = df.sort_values(
    ["vehicle_id", "tour_id", "start_time_parsed"]
)

tour_groups = df.groupby(
    ["vehicle_id", "tour_id"], dropna=False
)

tour_summary = tour_groups.agg(
    track_count=("track_id", "size"),
    first_start=("start_time_parsed", "min"),
    last_start=("start_time_parsed", "max"),
    first_home_base=("home_base", "first"),
    last_home_base=("home_base", "last"),
    unique_home_base_values=("home_base", "nunique"),
    unique_cids=("cid", "nunique"),
).reset_index()

tour_summary["span_hours"] = (
    tour_summary["last_start"] - tour_summary["first_start"]
).dt.total_seconds() / 3600

print("TOUR-GROUP SUMMARY")
print(tour_summary["track_count"].describe())
print("\nGroups with last track home_base=True:")
print(tour_summary["last_home_base"].value_counts(dropna=False))
print("\nGroups with first track home_base=True:")
print(tour_summary["first_home_base"].value_counts(dropna=False))
print("\nTour span in hours:")
print(tour_summary["span_hours"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))
print("\nGroups spanning more than 24 hours:",
      (tour_summary["span_hours"] > 24).sum())
print("\nGroups with multiple CID values:",
      (tour_summary["unique_cids"] > 1).sum())

TOUR-GROUP SUMMARY
count    13579.000000
mean        11.164151
std         36.696083
min          1.000000
25%          2.000000
50%          4.000000
75%          8.000000
max       2138.000000
Name: track_count, dtype: float64

Groups with last track home_base=True:
last_home_base
True     13413
False      166
Name: count, dtype: int64

Groups with first track home_base=True:
first_home_base
False    7750
True     5829
Name: count, dtype: int64

Tour span in hours:
count    13577.000000
mean        41.913687
std        299.001096
min          0.000000
50%          3.362407
75%         11.672158
90%         48.272106
95%         80.970257
99%        703.775322
max       5629.617855
Name: span_hours, dtype: float64

Groups spanning more than 24 hours: 1972

Groups with multiple CID values: 7446


In [32]:
# Identify candidate tour groups that pass the endpoint criterion
tour_keys = ["vehicle_id", "tour_id"]

valid_tour_keys = tour_summary.loc[
    tour_summary["last_home_base"].eq(True),
    tour_keys
]

excluded_tour_keys = tour_summary.loc[
    ~tour_summary["last_home_base"].eq(True),
    tour_keys
]

valid_index = pd.MultiIndex.from_frame(valid_tour_keys)
excluded_index = pd.MultiIndex.from_frame(excluded_tour_keys)
track_index = pd.MultiIndex.from_frame(df[tour_keys])

valid_mask = track_index.isin(valid_index)
excluded_mask = track_index.isin(excluded_index)

print("GROUP-LEVEL FILTER")
print(f"Total groups: {len(tour_summary):,}")
print(f"Retained groups: {len(valid_tour_keys):,}")
print(f"Excluded groups: {len(excluded_tour_keys):,}")
print(f"Group exclusion rate: {100 * len(excluded_tour_keys) / len(tour_summary):.2f}%")

print("\nTRACK-LEVEL IMPACT")
print(f"Total tracks: {len(df):,}")
print(f"Tracks retained: {valid_mask.sum():,}")
print(f"Tracks excluded: {excluded_mask.sum():,}")
print(f"Track exclusion rate: {100 * excluded_mask.sum() / len(df):.2f}%")

if "distance_km" in df.columns:
    total_distance = df["distance_km"].sum()
    excluded_distance = df.loc[excluded_mask, "distance_km"].sum()

    print("\nDISTANCE IMPACT")
    print(f"Total recorded distance: {total_distance:,.2f} km")
    print(f"Distance in excluded groups: {excluded_distance:,.2f} km")
    print(
        "Distance exclusion rate: "
        f"{100 * excluded_distance / total_distance:.2f}%"
        if total_distance > 0 else "Distance exclusion rate: undefined"
    )

GROUP-LEVEL FILTER
Total groups: 13,579
Retained groups: 13,413
Excluded groups: 166
Group exclusion rate: 1.22%

TRACK-LEVEL IMPACT
Total tracks: 151,598
Tracks retained: 135,342
Tracks excluded: 16,256
Track exclusion rate: 10.72%

DISTANCE IMPACT
Total recorded distance: 2,201,457.26 km
Distance in excluded groups: 379,016.13 km
Distance exclusion rate: 17.22%


In [33]:
# Summarize the excluded groups
excluded_summary = tour_summary.loc[
    ~tour_summary["last_home_base"].eq(True)
].copy()

# Attach track-level distance and duration totals
excluded_keys = pd.MultiIndex.from_frame(
    excluded_summary[["vehicle_id", "tour_id"]]
)

track_keys = pd.MultiIndex.from_frame(
    df[["vehicle_id", "tour_id"]]
)

excluded_tracks = df.loc[track_keys.isin(excluded_keys)].copy()

excluded_agg = excluded_tracks.groupby(
    ["vehicle_id", "tour_id"]
).agg(
    total_distance_km=("distance_km", "sum"),
    track_count=("track_id", "size"),
    first_start=("start_time_parsed", "min"),
    last_stop=("stop_time_parsed", "max"),
).reset_index()

excluded_agg["span_hours"] = (
    excluded_agg["last_stop"] - excluded_agg["first_start"]
).dt.total_seconds() / 3600

print("EXCLUDED GROUPS — DISTANCE DISTRIBUTION")
print(excluded_agg["total_distance_km"].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
))

print("\nTOP 20 EXCLUDED GROUPS BY DISTANCE")
display(
    excluded_agg.sort_values(
        "total_distance_km", ascending=False
    ).head(20)
)

EXCLUDED GROUPS — DISTANCE DISTRIBUTION
count      166.000000
mean      2283.229681
std       3572.902529
min          0.000000
50%        448.472000
75%       3128.169000
90%       7440.519000
95%       9612.668750
99%      14046.357750
max      20355.680000
Name: total_distance_km, dtype: float64

TOP 20 EXCLUDED GROUPS BY DISTANCE


,vehicle_id,tour_id,total_distance_km,track_count,first_start,last_stop,span_hours
160,159,13428,20355.680,706,2023-09-14 11:59:35.545000+00:00,2024-02-05 12:55:20.625000+00:00,3456.929189
157,156,13397,14760.022,245,2023-07-26 12:48:42.087000+00:00,2024-01-26 05:46:40.687000+00:00,4408.966278
76,80,12876,13662.077,266,2023-07-26 12:22:18.074000+00:00,2024-02-01 13:07:59.201000+00:00,4560.761424
79,82,12878,12576.763,285,2023-06-23 14:35:09.653000+00:00,2024-01-16 15:02:34.066000+00:00,4968.456781
140,140,13219,11358.931,306,2023-06-02 15:18:55.520000+00:00,2023-12-18 12:19:13.881000+00:00,4773.005100
48,55,9753,11255.950,393,2022-04-30 07:21:39.431000+00:00,2022-06-30 07:13:40.511000+00:00,1463.866967
109,112,13006,10671.150,283,2023-06-23 13:20:36.111000+00:00,2024-01-25 10:39:53.677000+00:00,5181.321546
72,76,12876,10371.879,296,2023-06-23 14:46:22.184000+00:00,2024-01-29 07:12:48.743000+00:00,5272.440711
107,110,13001,9649.528,302,2023-06-23 13:12:34.078000+00:00,2024-01-22 05:39:34.880000+00:00,5104.450223
74,78,12876,9502.091,353,2023-06-23 14:46:31.053000+00:00,2024-01-18 16:02:07.201000+00:00,5017.260041


In [34]:
# Summarize home-base flags inside the excluded groups
excluded_keys = pd.MultiIndex.from_frame(
    excluded_summary[["vehicle_id", "tour_id"]]
)
track_keys = pd.MultiIndex.from_frame(
    df[["vehicle_id", "tour_id"]]
)

excluded_tracks = df.loc[track_keys.isin(excluded_keys)].copy()
excluded_tracks = excluded_tracks.sort_values(
    ["vehicle_id", "tour_id", "start_time_parsed"]
)

home_base_diagnostic = excluded_tracks.groupby(
    ["vehicle_id", "tour_id"]
).agg(
    track_count=("track_id", "size"),
    home_base_true_count=("home_base", lambda s: s.eq(True).sum()),
    home_base_false_count=("home_base", lambda s: s.eq(False).sum()),
    total_distance_km=("distance_km", "sum"),
    first_start=("start_time_parsed", "min"),
    last_start=("start_time_parsed", "max"),
    last_home_base=("home_base", "last"),
).reset_index()

home_base_diagnostic["span_days"] = (
    home_base_diagnostic["last_start"]
    - home_base_diagnostic["first_start"]
).dt.total_seconds() / 86400

home_base_diagnostic["intermediate_home_base_count"] = (
    home_base_diagnostic["home_base_true_count"]
    - home_base_diagnostic["last_home_base"].eq(True).astype(int)
)

print("Excluded groups with at least one intermediate home-base track:",
      (home_base_diagnostic["intermediate_home_base_count"] > 0).sum())

print("\nHome-base flag counts within excluded groups:")
print(home_base_diagnostic[
    ["home_base_true_count", "home_base_false_count"]
].describe())

print("\nLongest excluded groups with intermediate home-base counts:")
display(
    home_base_diagnostic.sort_values(
        "total_distance_km", ascending=False
    ).head(20)
)

Excluded groups with at least one intermediate home-base track: 56

Home-base flag counts within excluded groups:
       home_base_true_count  home_base_false_count
count            166.000000             166.000000
mean               0.337349              97.590361
std                0.474236             140.539656
min                0.000000               1.000000
25%                0.000000               5.000000
50%                0.000000              20.000000
75%                1.000000             155.750000
max                1.000000             810.000000

Longest excluded groups with intermediate home-base counts:


,vehicle_id,tour_id,track_count,home_base_true_count,home_base_false_count,total_distance_km,first_start,last_start,last_home_base,span_days,intermediate_home_base_count
160,159,13428,706,0,706,20355.680,2023-09-14 11:59:35.545000+00:00,2024-02-05 12:50:50.513000+00:00,False,144.035590,0
157,156,13397,245,0,245,14760.022,2023-07-26 12:48:42.087000+00:00,2024-01-26 04:44:11.175000+00:00,False,183.663531,0
76,80,12876,266,0,266,13662.077,2023-07-26 12:22:18.074000+00:00,2024-02-01 12:57:20.304000+00:00,False,190.024331,0
79,82,12878,285,0,285,12576.763,2023-06-23 14:35:09.653000+00:00,2024-01-16 13:51:07.514000+00:00,False,206.969420,0
140,140,13219,306,0,306,11358.931,2023-06-02 15:18:55.520000+00:00,2023-12-18 12:19:04.585000+00:00,False,198.875105,0
48,55,9753,393,0,393,11255.950,2022-04-30 07:21:39.431000+00:00,2022-06-30 07:13:23.692000+00:00,False,60.994262,0
109,112,13006,283,0,283,10671.150,2023-06-23 13:20:36.111000+00:00,2024-01-25 10:26:10.357000+00:00,False,215.878869,0
72,76,12876,296,0,296,10371.879,2023-06-23 14:46:22.184000+00:00,2024-01-29 07:12:40.866000+00:00,False,219.684938,0
107,110,13001,302,0,302,9649.528,2023-06-23 13:12:34.078000+00:00,2024-01-22 05:39:25.217000+00:00,False,212.685314,0
74,78,12876,353,0,353,9502.091,2023-06-23 14:46:31.053000+00:00,2024-01-18 13:44:31.193000+00:00,False,208.956946,0


In [36]:

import pandas as pd
import numpy as np
from pathlib import Path

# --------------------------------------------------
# 1. Load a fresh copy of the original track dataset
# --------------------------------------------------
tracks = pd.read_csv("../data/processed/engineered_tracks.csv")

required_cols = [
    "vehicle_id", "tour_id", "track_id",
    "start_time", "stop_time", "distance_km", "home_base"
]

missing_cols = [c for c in required_cols if c not in tracks.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

tracks["start_time_parsed"] = pd.to_datetime(
    tracks["start_time"], errors="coerce", utc=True
)
tracks["stop_time_parsed"] = pd.to_datetime(
    tracks["stop_time"], errors="coerce", utc=True
)

tracks["distance_km"] = pd.to_numeric(
    tracks["distance_km"], errors="coerce"
)

# Use valid, positive per-track elapsed durations only.
tracks["track_duration_seconds"] = (
    tracks["stop_time_parsed"] - tracks["start_time_parsed"]
).dt.total_seconds()

tracks["valid_duration_seconds"] = tracks[
    "track_duration_seconds"
].where(
    tracks["track_duration_seconds"].ge(0)
)

# --------------------------------------------------
# 2. Sort tracks chronologically within each candidate tour
# --------------------------------------------------
tour_keys = ["vehicle_id", "tour_id"]

tracks = tracks.sort_values(
    tour_keys + ["start_time_parsed", "track_id"],
    na_position="last"
)

# Exclude records that cannot be assigned to a valid group.
assignable = tracks.dropna(subset=tour_keys).copy()

# --------------------------------------------------
# 3. Identify candidate tours by their final recorded track
# --------------------------------------------------
tour_groups = assignable.groupby(tour_keys, sort=False)

tour_summary = tour_groups.agg(
    track_count=("track_id", "size"),
    first_start=("start_time_parsed", "min"),
    last_start=("start_time_parsed", "max"),
    last_stop=("stop_time_parsed", "max"),
    total_distance_km=("distance_km", lambda s: s.sum(min_count=1)),
    valid_track_duration_seconds=("valid_duration_seconds", "sum"),
    valid_duration_track_count=("valid_duration_seconds", "count"),
    last_home_base=("home_base", "last"),
).reset_index()

# Endpoint rule agreed for the primary analysis.
valid_keys = tour_summary.loc[
    tour_summary["last_home_base"].eq(True), tour_keys
]

excluded_keys = tour_summary.loc[
    ~tour_summary["last_home_base"].eq(True), tour_keys
]

valid_index = pd.MultiIndex.from_frame(valid_keys)
excluded_index = pd.MultiIndex.from_frame(excluded_keys)
track_index = pd.MultiIndex.from_frame(assignable[tour_keys])

retained_tracks = assignable.loc[
    track_index.isin(valid_index)
].copy()

excluded_tracks = assignable.loc[
    track_index.isin(excluded_index)
].copy()

# --------------------------------------------------
# 4. Aggregate retained tracks into one row per tour
# --------------------------------------------------
retained_groups = retained_tracks.groupby(tour_keys, sort=False)

tours = retained_groups.agg(
    track_count=("track_id", "size"),
    start_time=("start_time_parsed", "min"),
    last_track_start=("start_time_parsed", "max"),
    end_time=("stop_time_parsed", "max"),
    total_distance_km=("distance_km", lambda s: s.sum(min_count=1)),
    total_valid_track_duration_seconds=(
        "valid_duration_seconds", "sum"
    ),
    valid_duration_track_count=(
        "valid_duration_seconds", "count"
    ),
    home_base_endpoint=("home_base", "last"),
).reset_index()

# Elapsed time across the whole tour, including gaps between tracks.
tours["elapsed_span_hours"] = (
    tours["end_time"] - tours["start_time"]
).dt.total_seconds() / 3600

# Driving/recorded-track time excludes gaps between tracks.
tours["recorded_track_hours"] = (
    tours["total_valid_track_duration_seconds"] / 3600
)

# Distance divided by summed valid track duration.
# This is not distance divided by elapsed tour span.
tours["distance_based_avg_speed_kmh"] = np.where(
    tours["total_valid_track_duration_seconds"] > 0,
    tours["total_distance_km"] * 3600
    / tours["total_valid_track_duration_seconds"],
    np.nan
)

# --------------------------------------------------
# 5. Add operational metadata for interpretation
# --------------------------------------------------
metadata_candidates = [
    "fleet_test_id", "freight_forwarder", "long_haul",
    "rest_area", "service_area_fuel", "industrial_area",
    "home_base", "cid"
]

metadata_cols = [
    c for c in metadata_candidates if c in retained_tracks.columns
]

if metadata_cols:
    metadata = retained_groups[metadata_cols].first().reset_index()
    tours = tours.merge(
        metadata, on=tour_keys, how="left", validate="one_to_one"
    )

# --------------------------------------------------
# 6. Save the dataset and exclusion audit
# --------------------------------------------------
output_dir = Path("data/processed/tour_level")
output_dir.mkdir(parents=True, exist_ok=True)

tours.to_csv(output_dir / "retained_tours.csv", index=False)
tour_summary.loc[
    ~tour_summary["last_home_base"].eq(True)
].to_csv(output_dir / "excluded_tour_groups.csv", index=False)

# Save track-level exclusion audit as well.
excluded_tracks.to_csv(
    output_dir / "excluded_tour_tracks.csv", index=False
)

# --------------------------------------------------
# 7. Report validation statistics
# --------------------------------------------------
print("TOUR DATASET CREATED")
print("-" * 50)
print(f"Input tracks:              {len(tracks):,}")
print(f"Assignable tracks:         {len(assignable):,}")
print(f"Retained candidate tours:  {len(tours):,}")
print(f"Excluded candidate tours:  {len(excluded_keys):,}")
print(f"Retained tracks:           {len(retained_tracks):,}")
print(f"Excluded tracks:           {len(excluded_tracks):,}")
print(f"Output shape:              {tours.shape}")

print("\nDISTANCE SUMMARY (km)")
print(tours["total_distance_km"].describe())

print("\nELAPSED TOUR SPAN (hours)")
print(tours["elapsed_span_hours"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
))

print("\nDISTANCE-BASED AVERAGE SPEED (km/h)")
print(tours["distance_based_avg_speed_kmh"].describe())

print("\nDATA QUALITY CHECKS")
print(
    "Duplicate vehicle-tour keys:",
    tours.duplicated(["vehicle_id", "tour_id"]).sum()
)
print(
    "Non-positive distances:",
    (tours["total_distance_km"] <= 0).sum()
)
print(
    "Missing distance totals:",
    tours["total_distance_km"].isna().sum()
)
print(
    "Missing/non-positive recorded duration:",
    tours["total_valid_track_duration_seconds"].le(0).sum()
    + tours["total_valid_track_duration_seconds"].isna().sum()
)
print(
    "Tours with incomplete track durations:",
    (
        tours["valid_duration_track_count"] < tours["track_count"]
    ).sum()
)

print("\nSaved files:")
print(output_dir / "retained_tours.csv")
print(output_dir / "excluded_tour_groups.csv")
print(output_dir / "excluded_tour_tracks.csv")


TOUR DATASET CREATED
--------------------------------------------------
Input tracks:              151,598
Assignable tracks:         151,598
Retained candidate tours:  13,413
Excluded candidate tours:  166
Retained tracks:           135,342
Excluded tracks:           16,256
Output shape:              (13413, 20)

DISTANCE SUMMARY (km)
count    13413.000000
mean       135.871254
std        324.057631
min          0.000000
25%          1.139000
50%         22.963000
75%        134.023000
max       7714.369000
Name: total_distance_km, dtype: float64

ELAPSED TOUR SPAN (hours)
count    13409.000000
mean        25.774672
std        160.263642
min        -30.042269
50%          4.383816
75%         11.884531
90%         46.055338
95%         71.288435
99%        268.745208
max       4024.936610
Name: elapsed_span_hours, dtype: float64

DISTANCE-BASED AVERAGE SPEED (km/h)
count    13408.000000
mean        24.236194
std         23.713300
min          0.000000
25%          3.727389
50%        

In [37]:

# 1. Tours with negative elapsed spans
negative_spans = tours.loc[
    tours["elapsed_span_hours"] < 0
].copy()

print("NEGATIVE ELAPSED SPANS:", len(negative_spans))
display(
    negative_spans[
        [
            "vehicle_id", "tour_id", "track_count",
            "start_time", "last_track_start", "end_time",
            "elapsed_span_hours", "total_distance_km"
        ]
    ].sort_values("elapsed_span_hours").head(20)
)

# 2. Tours with unusually high calculated average speeds
high_speed_tours = tours.loc[
    tours["distance_based_avg_speed_kmh"] > 120
].copy()

print("\nTOURS WITH DISTANCE-BASED SPEED > 120 KM/H:",
      len(high_speed_tours))

display(
    high_speed_tours[
        [
            "vehicle_id", "tour_id", "track_count",
            "total_distance_km",
            "total_valid_track_duration_seconds",
            "distance_based_avg_speed_kmh",
            "elapsed_span_hours",
            "valid_duration_track_count"
        ]
    ].sort_values(
        "distance_based_avg_speed_kmh", ascending=False
    ).head(20)
)

# 3. Tours with incomplete track durations
incomplete_duration_tours = tours.loc[
    tours["valid_duration_track_count"] < tours["track_count"]
].copy()

print("\nTOURS WITH INCOMPLETE TRACK DURATIONS:",
      len(incomplete_duration_tours))

display(
    incomplete_duration_tours[
        [
            "vehicle_id", "tour_id", "track_count",
            "valid_duration_track_count",
            "total_distance_km",
            "total_valid_track_duration_seconds",
            "elapsed_span_hours"
        ]
    ].head(20)
)


NEGATIVE ELAPSED SPANS: 1


,vehicle_id,tour_id,track_count,start_time,last_track_start,end_time,elapsed_span_hours,total_distance_km
12976,109,13000,1,2023-11-24 13:36:13.585000+00:00,2023-11-24 13:36:13.585000+00:00,2023-11-23 07:33:41.416000+00:00,-30.042269,0.0



TOURS WITH DISTANCE-BASED SPEED > 120 KM/H: 16


,vehicle_id,tour_id,track_count,total_distance_km,total_valid_track_duration_seconds,distance_based_avg_speed_kmh,elapsed_span_hours,valid_duration_track_count
10748,59,10761,1,90.390,497.633,653.903580,0.138231,1
8945,54,8954,3,115.206,735.415,563.955862,1.230461,3
3806,22,3813,5,184.096,1982.494,334.298918,1.571023,5
270,2,270,2,23.725,279.961,305.078207,1.890232,1
6546,33,6557,2,13.476,215.190,225.445420,0.703681,1
3981,22,3988,2,184.467,3406.007,194.973528,1.351436,2
2520,15,2526,8,309.329,5819.334,191.359424,2.852780,8
9653,55,9666,1,32.764,686.208,171.887241,0.190613,1
8784,54,8793,3,60.435,1416.950,153.545291,1.129002,3
11545,61,11559,3,151.675,3576.480,152.672460,3.895437,3



TOURS WITH INCOMPLETE TRACK DURATIONS: 237


,vehicle_id,tour_id,track_count,valid_duration_track_count,total_distance_km,total_valid_track_duration_seconds,elapsed_span_hours
12,1,12,23,22,822.566,59830.951,48.675751
39,1,39,588,585,118.251,590308.517,426.874708
52,1,52,20,19,73.210,15057.225,27.849292
55,1,55,436,434,57.945,281444.986,238.401351
192,2,192,33,32,17.806,16984.507,41.913160
261,2,261,1,0,0.042,0.000,NaN
270,2,270,2,1,23.725,279.961,1.890232
329,3,328,2,1,0.316,333.580,0.092661
388,3,387,1,0,0.196,0.000,NaN
465,3,464,8,7,389.039,22384.580,10.289216


In [38]:

from pathlib import Path
import numpy as np
import pandas as pd

# Copy the existing tour-level dataset
final_tours = tours.copy()

# 1. Timestamp consistency
final_tours["negative_elapsed_span"] = (
    final_tours["elapsed_span_hours"] < 0
)

# An elapsed span cannot be negative; preserve the flag and
# avoid using the impossible value in downstream analysis.
final_tours.loc[
    final_tours["negative_elapsed_span"],
    "elapsed_span_hours"
] = np.nan

# 2. Distance quality
final_tours["non_positive_distance"] = (
    final_tours["total_distance_km"].isna()
    | (final_tours["total_distance_km"] <= 0)
)

# 3. Duration completeness
final_tours["duration_incomplete"] = (
    final_tours["valid_duration_track_count"]
    < final_tours["track_count"]
)

final_tours["duration_unusable"] = (
    final_tours["total_valid_track_duration_seconds"].isna()
    | (final_tours["total_valid_track_duration_seconds"] <= 0)
)

# 4. Speed plausibility flag; review threshold, not automatic deletion
final_tours["high_distance_based_speed"] = (
    final_tours["distance_based_avg_speed_kmh"] > 120
)

# 5. Combined review flag
quality_flags = [
    "negative_elapsed_span",
    "non_positive_distance",
    "duration_incomplete",
    "duration_unusable",
    "high_distance_based_speed",
]

final_tours["requires_quality_review"] = (
    final_tours[quality_flags].any(axis=1)
)

# Save final dataset and a separate review list
output_dir = Path("data/processed/tour_level")
output_dir.mkdir(parents=True, exist_ok=True)

final_tours.to_csv(
    output_dir / "retained_tours_quality_checked.csv",
    index=False
)

final_tours.loc[
    final_tours["requires_quality_review"]
].to_csv(
    output_dir / "tour_quality_review.csv",
    index=False
)

print("FINAL TOUR DATASET")
print("-" * 45)
print("Total tours:", len(final_tours))
print("Negative elapsed spans:",
      final_tours["negative_elapsed_span"].sum())
print("Non-positive/missing distances:",
      final_tours["non_positive_distance"].sum())
print("Incomplete durations:",
      final_tours["duration_incomplete"].sum())
print("Unusable durations:",
      final_tours["duration_unusable"].sum())
print("High distance-based speeds:",
      final_tours["high_distance_based_speed"].sum())
print("Tours requiring review:",
      final_tours["requires_quality_review"].sum())

print("\nSaved:")
print(output_dir / "retained_tours_quality_checked.csv")
print(output_dir / "tour_quality_review.csv")


FINAL TOUR DATASET
---------------------------------------------
Total tours: 13413
Negative elapsed spans: 1
Non-positive/missing distances: 16
Incomplete durations: 237
Unusable durations: 5
High distance-based speeds: 16
Tours requiring review: 264

Saved:
data/processed/tour_level/retained_tours_quality_checked.csv
data/processed/tour_level/tour_quality_review.csv



## Tour-Level Dataset Construction and Quality Assessment

### Objective
Construct a tour-level dataset from the DT-CARGO recorded tracks dataset to support trip-level operational analysis and clustering. Each candidate tour is identified by the combination of `vehicle_id` and `tour_id`.

### 1. Tour Identification and Filtering
The original dataset contains **151,598 recorded tracks**. Grouping tracks by `(vehicle_id, tour_id)` produced **13,579 candidate tour groups**.

For the primary tour-level analysis, we retained groups whose last recorded track has `home_base = True`.

| Metric | Count |
|---|---:|
| Total candidate tour groups | 13,579 |
| Retained candidate tours | 13,413 |
| Excluded candidate groups | 166 |
| Group exclusion rate | 1.22% |
| Tracks retained | 135,342 |
| Tracks excluded | 16,256 |
| Track exclusion rate | 10.72% |

Although only 1.22% of candidate groups were excluded, they accounted for approximately **17.22% of the total recorded distance**. Several excluded groups contained unusually large numbers of tracks and spanned months. Therefore, the exclusion may introduce selection bias, particularly by underrepresenting some long-distance operations.

The retained groups are treated as candidate tours; the endpoint criterion does not independently verify every tour boundary.

### 2. Tour-Level Feature Engineering
Track-level records were aggregated into one row per retained `(vehicle_id, tour_id)` group. The resulting dataset contains **13,413 rows and 20 columns**.

Derived features include:
- Total recorded distance in kilometres.
- Start and end timestamps.
- Elapsed tour span in hours.
- Total valid recorded-track duration.
- Distance-based average speed in km/h.
- Track count and available operational metadata.

Distance-based average speed is calculated using total distance divided by the sum of valid recorded-track durations. It excludes gaps between recorded tracks and may differ from the dataset's original average-speed definition.

### 3. Data Quality Assessment
The initial quality checks identified:

| Quality issue | Tours affected |
|---|---:|
| Negative elapsed tour span | 1 |
| Non-positive or missing total distance | 16 |
| Incomplete track durations | 237 |
| Missing or non-positive total valid duration | 5 |
| Distance-based average speed above 120 km/h | 16 |
| Unique tours requiring quality review | 264 |

These flags can overlap. The 264 tours requiring review are not automatically removed. Their suitability depends on the features used in subsequent analyses.

The negative elapsed span was set to missing in the quality-checked dataset while retaining a flag for review. Original source timestamps were not modified.

### 4. Output Files
The following files were saved under `data/processed/tour_level/`:

- `retained_tours_quality_checked.csv` — retained candidate tours with quality flags.
- `tour_quality_review.csv` — tours requiring further inspection.
- `excluded_tour_groups.csv` — candidate groups excluded by the endpoint criterion.
- `excluded_tour_tracks.csv` — tracks belonging to excluded groups.

### 5. Limitations
1. The supplied `tour_id` values and endpoint flags do not guarantee that every candidate tour represents one independently verified real-world trip.
2. Excluding groups without a recorded home-base endpoint may introduce selection bias; the excluded groups accounted for 17.22% of total recorded distance.
3. Invalid or missing track durations can affect duration-based features and calculated average speeds.
4. Extreme values are flagged for investigation rather than automatically treated as errors.

### 6. Next Steps
The next stage is tour-level feature selection, preprocessing, standardization and PCA. Clustering models will then be compared using original standardized features and PCA representations. Evaluation will consider silhouette score, cluster sizes, stability and operational interpretability.
